# Reproducing Kohus et al. (2016) from CRCNS data set hc-7

Paper. Kohus Z, Káli S, Rovira-Esteban L, Schlingloff D, Papp O, Freund TF, Hájos N, Gulyás AI (2016).
Properties and dynamics of inhibitory synaptic communication within the CA3 microcircuits of pyramidal
cells and interneurons expressing parvalbumin or cholecystokinin. J Physiol.
[doi:10.1113/jp272231](https://doi.org/10.1113/jp272231). The claims file
(`claims/crcns-hc-7/10.1113-jp272231.yaml`) was written from the version of record (text extracted from
the PDF; the paper has no supplement and links no code).

Data set. CRCNS hc-7, [doi:10.6080/K0MK69T5](https://doi.org/10.6080/K0MK69T5), one zip
(`hc-7/live/Data.zip`) streamed from the public S3 bucket `crcnsarchive` and opened in place. Version:
CRCNS data sets are not versioned, so the version is recorded as `unversioned`, and the MD5 sum of
`Data.zip` from the data set's `checksums.md5` (`0fb0c04c9da6ce6f9b6b8281edd4484e`) is printed by the
first code cell, so that a later change of the file can be detected. The paper cites the data set by its
DOI only, without a version or date, and the bucket holds a single copy under `live/` (no `old/`), so
this copy is the one pinned. The data descriptor (`docs/crcns_hc-7_data_description.pdf`, version 0.8
of August 31, 2016) describes the files.

What the data set holds. Paired recordings (presynaptic membrane potential, postsynaptic current,
10 kHz) and the authors' per-sweep amplitude and success tables (`_EPA.txt`) for eight connection
types under the train, RIP, EPI and recovery protocols: 7 PV+BC to PC, 8 AAC to PC, 12 CCK+BC to PC,
7 CCK+DTI to PC, 14 PV+BC to PV+BC, 6 CCK+BC to CCK+BC, 4 CCK+DTI to CCK+DTI and 2 PV+BC to AAC pairs.
It holds none of the current-step recordings, connectivity tests, spontaneous IPSCs or field
potentials, and does not say which pipette solution each pair was recorded with.

Seeds. The only random numbers are the starting points of the model fits (section "Models"), drawn
from one generator with a fixed seed in that section.

| Claim | Status |
|---|---|
| fig2b-pv-cck-clusters, fig2b-pvbc-aac-clusters, cck-no-subclusters | blocked: current-step data missing |
| table1-pv-cck-features, table2-pvbc-aac-features | blocked: current-step data missing |
| fig3-cp-pv, fig3-cp-cck, fig3-cp-pv-cck | blocked: connectivity tests (unconnected pairs) missing |
| fig4b-in-pc-kruskal-wallis, fig4b-in-pc-posthoc | tested |
| fig4c-cckdti-pc-silent-single-ap | tested |
| fig7a-cscl-kinetics, fig7b-cscl-stp | blocked: pairs not labelled by pipette solution |
| fig5c-pv-rip-depression, fig5c-cck-rip-facilitation | tested |
| fig5db-recovery | tested |
| fig6-pv-trains-depression, fig6-cckbc-pc-trains | tested |
| fig6-cckbc-async-onset, cckdti-async-onset | tested |
| cckdti-pc-transmission | tested |
| async-charge-epi-vs-160, decay-epi-vs-160 | tested |
| fig8-pvbc-pvbc-uipsc | tested |
| fig8-cck-pairs-uipsc, fig8-cckbc-vs-cckdti-pairs | blocked: no single-AP recordings of the CCK+ pairs (reading d shown) |
| fig9-pvbc-pvbc-depression, fig9-pvbc-pvbc-transmission-probability, fig9-pvbc-pvbc-recovery-11th | tested |
| fig10-cck-cck-heterogeneity, fig10-cck-cck-async | tested |
| fig10bd-be-async-charge | blocked: no 320 Hz trains of the CCK+ pairs (10 to 160 Hz parts shown) |
| fig10-cck-cck-frequency | tested |
| fig11-model-pv-udr, fig11-model-cck-facilitation, table4-model-parameters, fig11c-parameter-differences, fig11-model-prediction | tested |
| fig12-sipsc-kinetics, sipsc-continuum | blocked: spontaneous IPSC data missing |
| decay-pc-vs-interneuron | blocked: depends on fig8-cck-pairs-uipsc (PV+BC part shown) |
| fig13-pvbc-pc-recovery-halftime | tested |
| fig13a-swr-refractory, fig13c-optogenetic-swr | blocked: field potential data missing |

In [ ]:
import collections, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
import hc7

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 400); pd.set_option("display.max_columns", 40)
t_start = time.time()
z = hc7.open_zip()
md5 = hc7.checksums()
print("data set", hc7.DATASET, "version", hc7.VERSION, "doi", hc7.DATASET_DOI)
print("checksums.md5 of the files read:", {k: v for k, v in md5.items() if k in ("Data.zip",)})
EX = hc7.experiments_table(z)
RT = hc7.recordings_table(z)
print(len(RT), "recordings (.hda headers) in", RT.folder.nunique(), "connection folders;",
      RT.groupby("type").pair.nunique().to_dict())

Every sweep of every recording is read once (about 1.3 GB uncompressed) and kept in memory as float32;
this takes a few minutes.

In [ ]:
# Read every sweep of every recording with raw data (about 1.3 GB uncompressed; a few minutes).
RECS = {}
for r in RT[RT.n_ch1 > 0].itertuples():
    RECS[r.path] = hc7.load_recording(z, r.path, min(r.n_ch1, r.n_ch2), r.rate)
print(len(RECS), "recordings loaded in", round(time.time() - t_start), "s")

## Data check

The checks below compare the zip with the data descriptor and with `Experiments_description.txt`, the
table of recordings per pair at the zip's root.

In [ ]:
# Data check: what the zip holds, against the data descriptor and Experiments_description.txt.
rows = []
for r in RT.itertuples():
    rec = RECS.get(r.path)
    proto = hc7.protocol_of(r.sweep_ms, rec) if rec else None
    cls = collections.Counter(hc7.classify(a) for a in rec.ap).most_common(1)[0][0] if rec else None
    m = EX[EX.entry == r.base + "_EPA.txt"]
    rows.append(dict(type=r.type, pair=r.pair, base=r.base, path=r.path, n_sweeps=r.n_sweeps, n_bins=min(r.n_ch1, r.n_ch2),
                     sweep_ms=r.sweep_ms, rate=r.rate, protocol=proto, ap_class=cls, has_epa=r.has_epa,
                     pre_channel=rec.pre_channel if rec else None,
                     desc=";".join(f"Pair {b} {c}" for b, c in zip(m.pair, m.protocol))))
S = pd.DataFrame(rows)
print("sample rates (Hz):", S.rate.unique(), "| sweep lengths (ms) by protocol:")
print(S.groupby("protocol").sweep_ms.unique().to_string())
print("\nrecordings by connection type and protocol (all folders):")
print(pd.crosstab(S.type, S.protocol))
print("\npairs per connection type in the zip:", S.groupby("type").pair.nunique().to_dict())
print("pairs per connection type in Experiments_description.txt:", EX.groupby("connection").pair.nunique().to_dict())
print("\nrecordings without sweeps or without an _EPA table:")
print(S[(S.n_bins == 0) | ~S.has_epa][["type", "pair", "base", "n_sweeps", "n_bins", "protocol", "has_epa"]].to_string())
# the protocol from the sweep length against the AP pattern of the presynaptic channel
chk = S[S.n_bins > 0].copy()
chk["agree"] = [(p == c) or (p == "Recovery" and c == "other (9 APs)") for p, c in zip(chk.protocol, chk.ap_class)]
print("\nprotocol by sweep length agrees with the modal AP pattern in", int(chk.agree.sum()), "of", len(chk),
      "recordings (recovery sweeps carry a ninth AP about 300 ms after the second burst in 6 of 7 intervals); "
      "the others, with extra APs in most sweeps:")
print(chk[~chk.agree][["type", "pair", "base", "protocol", "ap_class"]].to_string())
# description labels against the data
lab = S[S.desc != ""].copy()
lab["desc_protocol"] = lab.desc.str.split(" ").str[2:].str.join(" ")
print("\nrecordings whose Experiments_description.txt label differs from the sweep length and AP pattern:")
print(lab[[str(p) not in str(d) for p, d in zip(lab.protocol, lab.desc)]][["type", "pair", "base", "protocol", "desc"]].to_string())
print("recordings in the zip not named in Experiments_description.txt:", int((S.desc == "").sum()))
print(S[S.desc == ""][["type", "pair", "base", "protocol"]].to_string())
named = EX[EX.entry.str.endswith(".txt")]
missing = named[~named.entry.str[:-8].isin(S.base)]
print("files named in Experiments_description.txt that are not in the zip:")
print(missing.to_string())
print("'No release' entries:", EX[EX.entry == "No release"].groupby("connection").size().to_dict())
# folder numbering against the description's numbering
fmap = (S[S.desc != ""].assign(dpair=S.desc.str.extract(r"Pair (\d+)")[0])
        .groupby(["type", "pair"]).dpair.agg(lambda x: sorted(set(x), key=int)))
print("\nfolder PairNN -> Experiments_description.txt pair number(s):")
print(fmap.to_string())
# duplicates: identical first postsynaptic sweeps
import hashlib
h = {p: hashlib.md5(r.post[0].tobytes()).hexdigest() for p, r in RECS.items()}
dups = pd.Series(h)[pd.Series(h).duplicated(keep=False)]
print("\nrecordings with identical first postsynaptic sweep:", dups.index.tolist())
# ranges and units of the channels used
pre_rng = np.array([[r.pre_max.min(), r.pre_max.max()] for r in RECS.values()])
post_rng = np.array([r.post_vrange for r in RECS.values()])
print("presynaptic channel: AP peaks", np.round(pre_rng.min(), 1), "to", np.round(pre_rng.max(), 1), "mV")
print("postsynaptic channel: current", np.round(post_rng[:, 0].min(), 1), "to", np.round(post_rng[:, 1].max(), 1), "pA")

What the check shows (each statement rests on the output above):

- The sample rate is 10 kHz in every header. Each protocol has its own sweep length (RIP and EPI share
  2411 to 2419 ms and are told apart by the AP count), and the protocol given by the sweep length agrees
  with the AP pattern of the presynaptic channel in every recording, apart from three with extra APs in
  most sweeps.
- The presynaptic channel is identified from the data (APs above a narrow resting range); the
  headers' channel units do not say which channel is which, as the descriptor warns.
- Pair counts in the zip: 7 PV+BC to PC (the paper reports 8 with high KCl), 8 AAC to PC (9), 12 CCK+BC
  to PC (14), 7 CCK+DTI to PC (8), 14 PV+BC to PV+BC (16, 13 with trains), 6 CCK+BC to CCK+BC (6), 4
  CCK+DTI to CCK+DTI (4), 2 PV+BC to AAC (2). The descriptor's folder tree agrees with the zip.
  `Experiments_description.txt` lists 8 PV+BC to PC pairs (its pair 4 has a RIP file that is not in the
  zip) and no PV+BC to AAC pairs.
- The folder numbering (`PairNN`) differs from the numbering of `Experiments_description.txt` for the
  IN to PC types (mapping printed above); the match is by file name.
- `Experiments_description.txt` mislabels two recordings (PV+BC to PV+BC Pair10 `s130807-0205`, listed as
  5 Hz, is a 10 Hz train; CCK+DTI to PC Pair04 `s150222-0305`, listed as 40 Hz, is a 10 Hz train), names
  eight files that are not in the zip (three of them misdated, `s1508..` for `s1308..`), and leaves 20
  recordings of the zip unnamed. AAC to PC pair 7 is listed with one file for both Recovery and 40 Hz;
  CCK+DTI to PC pair 7 with one file for both RIP and EPI.
- Three recordings have a header but no sweeps (CCK+BC to PC Pair07 160 and 40 Hz, CCK+DTI to PC Pair01
  Recovery); three have sweeps but no `_EPA` table.
- No two recordings have the same first postsynaptic sweep.
- The postsynaptic current ranges over -4000 to 999.5 pA, so some sweeps reach the recording range;
  AP peaks reach up to 100 mV.

In [ ]:
# Pair table: for each connection and folder pair, the recording of each protocol (by sweep length and
# AP count). A pair with two recordings of one protocol keeps the first and the second is reported.
PAIRS = collections.defaultdict(dict)
extra = []
for r in S[S.n_bins > 0].sort_values("base").itertuples():
    if r.protocol in PAIRS[(r.type, r.pair)]:
        extra.append((r.type, r.pair, r.protocol, r.base))
        continue
    PAIRS[(r.type, r.pair)][r.protocol] = r.path
print("second recordings of a protocol in one pair (not used):", extra)
EPA = {r.path: hc7.read_epa(z, r.path) for r in S[S.has_epa].itertuples()}
IN_PC = ["PV+BC to PC", "AAC to PC", "CCK+BC to PC", "CCK+DTI to PC"]
IN_IN = ["PV+BC to PV+BC", "CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI", "PV+BC to AAC"]
PROTOS = ["RIP", "EPI", "Recovery"] + [f"{f} Hz" for f in hc7.FREQS]
cov = pd.DataFrame({k: {p: (p in v) for p in PROTOS} for k, v in PAIRS.items()}).T.astype(int)
print("protocols with raw data per pair:")
print(cov.to_string())
print("\npairs with each protocol, by type:")
print(cov.groupby(level=0).sum().loc[IN_PC + IN_IN].to_string())
# EPA tables against the raw sweeps: the tables give one row per sweep, numbered; sweeps of the raw data
# missing from the table, and tables whose mean row is not the mean of the per-sweep amplitudes
bad = []
for p, e in EPA.items():
    n_raw = len(RECS[p].post) if p in RECS else 0
    blocks = range(e.mean_amp.shape[0])
    dm = max(np.nanmax(np.abs(np.nanmean(e.amp[e.block == b], 0) - e.mean_amp[b])) for b in blocks)
    dt = max(np.nanmax(np.abs(np.nanmean(e.success[e.block == b], 0) - e.tp[b])) for b in blocks)
    incons = int(((e.success == 0) & (e.amp > 0)).sum() + ((e.success == 1) & (e.amp == 0)).sum())
    if len(e.sweeps) != n_raw or dm > 0.01 or dt > 0.01 or incons or np.isnan(e.amp).any():
        bad.append(dict(rec=p[11:], raw_sweeps=n_raw, table_sweeps=len(e.sweeps), mean_row_diff_pA=round(dm, 2),
                        tp_row_diff=round(dt, 3), success_amp_mismatch=incons, blank_amp_cells=int(np.isnan(e.amp).sum())))
print(f"\n{len(EPA)} _EPA tables; {len(bad)} with a difference from the raw sweeps or within the table:")
print(pd.DataFrame(bad).to_string())
ar = pd.DataFrame([dict(type=S.set_index("path").type[p], protocol=S.set_index("path").protocol[p]) for p, e in EPA.items()
                   if e.ar_first is not None])
print("\ntables whose mean row carries the code 'AR' from some AP on:", len(ar))
print(ar.groupby(["type", "protocol"]).size().to_string())
nolab = [p[11:] for p, e in EPA.items() if np.isnan(e.mean_amp).all()]
print("tables with no numeric value in the mean row:", nolab)
print("tables where success and amplitude disagree in some cell:", [b["rec"] for b in bad if b["success_amp_mismatch"]])

- The `_EPA` tables mostly agree with themselves: their mean row is the mean of the per-sweep
  amplitudes, failures carry amplitude 0. The exceptions are listed above: tables with fewer sweeps than
  the raw data (sweeps left out without a note), blank cells, a few mean rows that differ from their
  own per-sweep values, and four AAC to PC tables where success and amplitude disagree in some cells.
  In 21 tables (the EPI tables of every CCK+BC and CCK+DTI to PC pair, and four CCK+DTI to PC 160 Hz
  tables) the mean row is cut off by the code `AR` from some AP on, and two CCK+BC to PC 160 Hz tables
  (`s140619-0204`, `s140619-0105`) hold column labels in place of the mean values; the analysis uses the
  per-sweep rows, not the mean row. The descriptor does not
  describe `AR`; its position matches the onset of asynchronous release the paper reports (section
  "Asynchronous release"). The descriptor's "Relative ampl." block holds amplitudes in pA, not
  relative values.
- RIP bursts are at 222 Hz (4.5 ms between APs), not 160 Hz; the EPI burst is at 147 Hz then 303 Hz,
  not 160 and 320 Hz; the single AP precedes the RIP and EPI bursts by 1.3 s, not 1 s (AP times printed
  in the section "Single-AP measures"). The 10-AP trains are at their nominal rates.
- Recovery sweeps cycle through the intervals 1200, 800, 500, 300, 200, 100 and 50 ms in that order,
  read from the AP times; the descriptor gives the intervals but not the order, and the `_EPA` table
  groups sweeps k, k+7, k+14, ... in its k-th block without naming the interval.

## Blocked claims

Decided before the analysis, from the file list and the descriptor:

- fig2b-pv-cck-clusters, fig2b-pvbc-aac-clusters, cck-no-subclusters, table1-pv-cck-features,
  table2-pvbc-aac-features: the 800 ms current-step responses of the 67 interneurons are not in the data
  set (`data_missing`).
- fig3-cp-pv, fig3-cp-cck, fig3-cp-pv-cck: connection probability needs the tested pairs that were not
  connected; the data set holds connected pairs only (`data_missing`).
- fig7a-cscl-kinetics, fig7b-cscl-stp: the pairs are not labelled by pipette solution, so CsCl pairs
  cannot be told from high KCl pairs (`metadata_insufficient`). All IN to PC pairs are treated as high
  KCl pairs elsewhere, as each type has no more pairs than the paper's high KCl count. This is an
  assumption: the check in the section "Single-AP measures" (holding current, rise time and latency per
  pair) finds no split within a type, but cannot rule out a mixture, and the outcomes on IN to PC pairs
  are marked as resting on it.
- fig12-sipsc-kinetics, sipsc-continuum: no spontaneous IPSC recordings (`data_missing`).
- fig13a-swr-refractory, fig13c-optogenetic-swr: no field potential recordings (`data_missing`).

Decided once the pair table was made: the CCK+ to CCK+ pairs were recorded only with 10, 40 and 160 Hz
trains, so none of the claims file's single-AP readings (M3 a to c) exists for them, and they have no
320 Hz train. fig8-cck-pairs-uipsc and fig8-cckbc-vs-cckdti-pairs are blocked (`data_missing`), and so
is decay-pc-vs-interneuron, which depends on fig8-cck-pairs-uipsc; fig10bd-be-async-charge is blocked
(`data_missing`) for its 320 Hz parts. Their other parts are shown.

## Single-AP measures

In [ ]:
# Single-AP uIPSC measures per pair (claims M1 to M3). Readings of the single AP (M3): (a) the single AP
# 1.3 s before the RIP burst (column T01S01 of the RIP table); (b) the ten APs of the 1 Hz train; (c) both
# pooled. Reading (d), for pairs with neither (the CCK+ interneuron pairs, recorded only with 10, 40 and
# 160 Hz trains): the first AP of every 10-AP train at 40 Hz or slower, which follows at least 1 s without
# APs (paper, Methods, "repeated 10 times, with 1 s intervals between the two trains"), the decay fit
# ending 1 ms before the second AP; not a reading of the claims file, used only where (a) to (c) cannot
# be run, and shown for the other pairs for comparison. Failures (M2): (a) the authors' per-sweep success in the _EPA table;
# (b) detection on the raw sweeps, peak above 3 SD of the pre-AP baseline.
def trials(path, ap_cols):
    """(sweep index, AP time, success a) for the given AP columns of a recording, over the sweeps that the
    _EPA table lists and in which the AP is present."""
    rec, e = RECS[path], EPA.get(path)
    proto = S.set_index("path").protocol[path]
    times, present = hc7.stim_template(rec, proto)
    out = []
    if e is None:
        return out
    for row, sw in enumerate(e.sweeps):
        s = sw - 1
        if s >= len(rec.post):
            continue
        for c in ap_cols:
            if c < len(times) and present[s, c] and np.isfinite(e.success[row, c]):
                out.append((s, times[c], e.success[row, c], path))
    return out

def single_trials(key, reading):
    pr = PAIRS[key]
    tr = []
    if reading in ("a", "c") and "RIP" in pr:
        tr += trials(pr["RIP"], [0])
    if reading in ("b", "c") and "1 Hz" in pr:
        tr += trials(pr["1 Hz"], range(10))
    if reading == "d":
        for p in [f"{f} Hz" for f in hc7.FREQS if 1 < f <= 40]:
            if p in pr:
                tr += trials(pr[p], [0])
    return tr

def measure_pair(key, reading, failures="a", decay_win=hc7.DECAY_WINS[0]):
    tr = single_trials(key, reading)
    if not tr:
        return None
    ys = []
    for s, t, ok, p in tr:
        tt, y = hc7.snippets(RECS[p], [s], [t])
        ys.append(y[0])
    ys = np.array(ys)
    succ = np.array([ok for *_, ok, _ in [(a, b, c, d) for a, b, c, d in tr]]) if failures == "a" else hc7.detect_success(tt, ys)
    fit_win = decay_win
    if reading == "d":   # end the decay fit before the second AP of the fastest train used
        isi = min(1000 / int(S.set_index("path").protocol[p].split()[0]) for *_, p in tr)
        fit_win = min(decay_win, isi - 1 - 1.5)
    m = hc7.single_ap_measures(tt, ys, succ, fit_win)
    m.update(type=key[0], pair=key[1], reading=reading, failures=failures, decay_win=decay_win, fit_win=fit_win)
    return m

SINGLE = []
for key in PAIRS:
    for reading in "abcd":
        for failures in "ab":
            for dw in hc7.DECAY_WINS:
                m = measure_pair(key, reading, failures, dw)
                if m:
                    SINGLE.append(m)
SINGLE = pd.DataFrame(SINGLE)
print("pairs with single-AP measures, by reading of M3 (failures a, decay window 20 ms):")
print(SINGLE[(SINGLE.failures == "a") & (SINGLE.decay_win == 20)].groupby(["type", "reading"]).pair.nunique().unstack().to_string())
cols = ["rise", "tau", "latency", "amplitude", "potency", "tp"]
print("\nmedians over pairs, failures a, decay window 20 ms:")
print(SINGLE[(SINGLE.failures == "a") & (SINGLE.decay_win == 20)].groupby(["type", "reading"])[cols].median().round(2).to_string())

# Clock check: the uIPSC follows the presynaptic AP. Mean peak of the baseline-subtracted current in the
# response window after the single AP before RIP (M3 a), against the same window placed 100 ms earlier,
# where no AP occurs; per connection type, median over pairs.
rows = []
for key in sorted(k for k in PAIRS if k[0] in ["PV+BC to PC", "AAC to PC", "CCK+BC to PC", "CCK+DTI to PC"]):
    tr = single_trials(key, "a")
    if not tr:
        continue
    for shift in (0.0, -100.0):
        tt, ys = hc7.snippets(RECS[tr[0][3]], [s for s, *_ in tr], [t + shift for _, t, *_ in tr])
        k = hc7.kinetics(tt, np.nanmean(ys, 0))
        rows.append(dict(type=key[0], pair=key[1], shift_ms=shift, peak=k["peak"]))
CLK = pd.DataFrame(rows).pivot_table(index=["type", "pair"], columns="shift_ms", values="peak")
print("\nclock check, peak of the averaged current (pA) after the AP (0) and 100 ms before it (-100):")
print(CLK.groupby(level=0).median().round(1).to_string())
print("pairs with a larger response at the AP than 100 ms before:", int((CLK[0.0] > CLK[-100.0]).sum()), "of", len(CLK))
# recording days, the only grouping above the pair that the files give (no animal or slice identifiers)
days = S[S.n_bins > 0].assign(day=S.base.str[1:7]).groupby(["type", "pair"]).day.first()
print("\npairs per recording day (file name sYYMMDD), by type:")
print(days.reset_index().groupby(["type", "day"]).size().unstack(0).fillna(0).astype(int).to_string())

The clock check: in all 30 IN to PC pairs with a single AP before RIP, the averaged current after the
AP is larger than in the same window 100 ms earlier (medians 28.5 to 226.1 pA against 2.0 to 7.3 pA),
so the postsynaptic current shares the clock of the presynaptic APs. The files carry no animal or slice
identifiers; the recording day in the file name is the only grouping above the pair, and most days
contribute one to five pairs of one type. Results are not broken down by day.

Example AP times (ms) of one sweep of each burst protocol, PV+BC to PC Pair02:

In [ ]:
for proto in ("RIP", "EPI", "10 Hz"):
    p = PAIRS[("PV+BC to PC", "Pair02")][proto]
    t, _ = hc7.stim_template(RECS[p], proto)
    print(proto, np.round(t[:8], 1), "... intervals (ms):", np.round(np.unique(np.round(np.diff(t[1:] if proto != "10 Hz" else t[:10]), 1)), 1))
# the _EPA column labels against the AP times of the same recording (RIP: T01S01 is the single AP)
p = PAIRS[("PV+BC to PC", "Pair02")]["RIP"]
t, _ = hc7.stim_template(RECS[p], "RIP")
print("RIP _EPA columns and AP times (ms):", list(zip(EPA[p].columns, np.round(t, 1))))
p = PAIRS[("PV+BC to PC", "Pair02")]["10 Hz"]
t, _ = hc7.stim_template(RECS[p], "10 Hz")
print("10 Hz _EPA columns and AP times (ms):", list(zip(EPA[p].columns, np.round(t, 1))))

Pipette solution. The paper recorded most IN to PC pairs with high KCl in both cells and some with low
KCl (presynaptic) and high CsCl with QX-314 (postsynaptic); CsCl gave longer rise times and latencies for
all three perisomatic types (Results, paragraph 3 of the section on uIPSCs onto PCs). The files do not
say which solution a pair was recorded with. The table below gives, per IN to PC pair, the recording day,
the rise time and latency of the single-AP uIPSC (M3 a, M2 a) and the holding current of the PC at -60 mV
(the median raw current over the first 50 ms of the RIP sweeps, before any AP), which Cs+ and QX-314 in
the pipette would be expected to shift by blocking K+ conductances. It is a check for a mixture, not a
label.

In [ ]:
# Per IN to PC pair: recording day, single-AP rise and latency (M3 a, M2 a), PC holding current (pA) in the RIP sweeps.
rows = []
for key in sorted(k for k in PAIRS if k[0] in IN_PC):
    p = PAIRS[key].get("RIP") or next(iter(PAIRS[key].values()))
    rec = RECS[p]
    hold = float(np.median(rec.post[:, :int(50 * rec.rate / 1e3)]))
    m = SINGLE[(SINGLE.type == key[0]) & (SINGLE.pair == key[1]) & (SINGLE.reading == "a") & (SINGLE.failures == "a") & (SINGLE.decay_win == 20)]
    rows.append(dict(type=key[0], pair=key[1], day=p.split("/")[-1][1:7], holding_pA=round(hold, 1),
                     rise=round(float(m.rise.iloc[0]), 3) if len(m) else np.nan,
                     latency=round(float(m.latency.iloc[0]), 3) if len(m) else np.nan))
SOL = pd.DataFrame(rows)
print(SOL.sort_values(["type", "day"]).to_string())
print("\nby type: range of holding current, rise and latency; Spearman of rise and of latency with the holding current")
for g, d in SOL.groupby("type"):
    d = d.dropna()
    rr, pr = stats.spearmanr(d.holding_pA, d.rise) if len(d) > 2 else (np.nan, np.nan)
    rl, pl = stats.spearmanr(d.holding_pA, d.latency) if len(d) > 2 else (np.nan, np.nan)
    print(f"  {g} ({len(d)} pairs): holding {d.holding_pA.min()} to {d.holding_pA.max()} pA, rise {d.rise.min()} to {d.rise.max()} ms, "
          f"latency {d.latency.min()} to {d.latency.max()} ms; rho rise {rr:.2f} (p {pr:.2g}), rho latency {rl:.2f} (p {pl:.2g})")
# Sensitivity (round-2 verifier): rise and latency across PV+BC, AAC and CCK+BC to PC with all CCK+BC to PC pairs, and
# with only the half with the more negative holding current, where by the correlation above latencies are shortest.
cck = SOL[SOL.type == "CCK+BC to PC"].dropna()
for label, keep in (("all CCK+BC to PC pairs", cck), ("CCK+BC to PC pairs with holding current at or below the median", cck[cck.holding_pA <= cck.holding_pA.median()])):
    d = pd.concat([SOL[SOL.type.isin(["PV+BC to PC", "AAC to PC"])].dropna(), keep])
    kw = {q: stats.kruskal(*[d[d.type == g][q] for g in ["PV+BC to PC", "AAC to PC", "CCK+BC to PC"]]).pvalue for q in ("rise", "latency")}
    print(f"{label} (n={len(keep)}): CCK+BC median latency {keep.latency.median():.3f} ms, rise {keep.rise.median():.3f} ms; "
          f"Kruskal-Wallis p rise {kw['rise']:.2g}, latency {kw['latency']:.2g}")


Within each type the holding currents spread continuously (PV+BC to PC -358.4 to 14.2 pA, AAC to PC
-376.0 to 9.8 pA, CCK+BC to PC -159.7 to 23.4 pA), with no gap that would mark two groups, and the rise
times and latencies show no split either. Latency rises with the holding current in CCK+BC to PC pairs
(rho 0.58, p 0.048) and not significantly in the other types. This is the one sign in the data against the
assumption: CsCl lengthens latency, so if the pairs with less negative holding current were CsCl pairs,
they would lengthen the CCK+BC latencies and push toward the latency difference of fig4b. With only the 6
CCK+BC to PC pairs at or below the type's median holding current, the Kruskal-Wallis tests across PV+BC,
AAC and CCK+BC to PC give p 0.26 for rise and 0.14 for latency (0.021 and 0.0084 with all 12 pairs), so
the fig4b differences in rise and latency depend on the other 6 pairs. With half the pairs the tests also
lose power, so this does not show that the difference is absent. None of this shows that every pair is a
high KCl pair: a mixture with CsCl pairs cannot be ruled out from these files. The outcomes on IN to PC
pairs rest on that assumption (`assumed` in the record).

## fig4b-in-pc-kruskal-wallis

> uIPSCs evoked in PCs by PV+BCs, AACs and CCK+BCs differ in 10 to 90% rise time, latency, amplitude,
> synaptic potency and transmission probability.

Reproduced when, under one reading of M2 and M3, all five Kruskal-Wallis tests give p < 0.05.

In [ ]:
# fig4b-in-pc-kruskal-wallis and fig4b-in-pc-posthoc: PV+BC, AAC and CCK+BC to PC pairs.
G3 = ["PV+BC to PC", "AAC to PC", "CCK+BC to PC"]
MEAS = ["rise", "latency", "amplitude", "potency", "tp", "tau"]
def sel(types, reading, failures, dw=20.0):
    return SINGLE[SINGLE.type.isin(types) & (SINGLE.reading == reading) & (SINGLE.failures == failures) & (SINGLE.decay_win == dw)]
kw_rows = []
for reading in "abc":
    for failures in "ab":
        d = sel(G3, reading, failures)
        row = dict(M3=reading, M2=failures, n=" / ".join(str((d.type == g).sum()) for g in G3))
        for m in MEAS:
            groups = [d[d.type == g][m].dropna() for g in G3]
            row[m] = stats.kruskal(*groups).pvalue if all(len(x) > 1 for x in groups) else np.nan
        row["n_sig_of_5"] = sum(row[m] < 0.05 for m in MEAS[:5])
        kw_rows.append(row)
KW = pd.DataFrame(kw_rows)
print("Kruskal-Wallis p across PV+BC, AAC and CCK+BC to PC (n = pairs PV+BC / AAC / CCK+BC); tau (decay) is shown, not judged:")
print(KW.to_string(float_format=lambda x: f"{x:.3g}"))
print("\nmedians by type (M3 a, M2 a):")
print(sel(G3, "a", "a").groupby("type")[MEAS].median().loc[G3].round(3).to_string())
print("ranges by type (M3 a, M2 a):")
print(sel(G3, "a", "a").groupby("type")[MEAS].agg(["min", "max"]).loc[G3].round(2).to_string())

**Result: reproduced.** With the single AP before RIP (M3 a), all five measures differ across the three
types at p < 0.05, with the authors' failures (M2 a; p 0.0191, 0.00842, 0.00405, 0.00624, 0.00951) and
with detected failures (M2 b; p 0.00653 to 0.0179). With the 1 Hz train (M3 b, 4, 3 and 3 pairs) no
measure differs; pooled (M3 c) three or four do. The decay time constant does not differ (p 0.122 under
M3 a, M2 a), as the paper reports. Pairs: 7 PV+BC, 7 AAC (one AAC pair has no RIP recording) and 12
CCK+BC, against the paper's 8, 9 and 14.

## fig4b-in-pc-posthoc

> PV+BC and AAC uIPSCs onto PCs rise faster, have shorter latency and larger amplitude than CCK+BC
> uIPSCs; AAC uIPSCs have larger potency, and higher transmission probability, than CCK+BC uIPSCs.

Eight two-sided Mann-Whitney comparisons, Bonferroni factor 3; reproduced when all eight hold under one
reading, partially when at least five do.

In [ ]:
# Post hoc: two-sided Mann-Whitney U, Bonferroni factor 3 (claims M6). Expected directions.
COMP = [("rise", "PV+BC to PC", "CCK+BC to PC", "<"), ("rise", "AAC to PC", "CCK+BC to PC", "<"),
        ("latency", "PV+BC to PC", "CCK+BC to PC", "<"), ("latency", "AAC to PC", "CCK+BC to PC", "<"),
        ("amplitude", "PV+BC to PC", "CCK+BC to PC", ">"), ("amplitude", "AAC to PC", "CCK+BC to PC", ">"),
        ("potency", "AAC to PC", "CCK+BC to PC", ">"), ("tp", "AAC to PC", "CCK+BC to PC", ">")]
def posthoc(d, comps, factor):
    rows = []
    for m, g1, g2, sign in comps:
        x, y = d[d.type == g1][m].dropna(), d[d.type == g2][m].dropna()
        p = stats.mannwhitneyu(x, y, alternative="two-sided").pvalue if len(x) and len(y) else np.nan
        dirn = (x.median() < y.median()) if sign == "<" else (x.median() > y.median())
        rows.append(dict(measure=m, g1=g1, g2=g2, expected=sign, med1=round(x.median(), 3), med2=round(y.median(), 3),
                         p=p, p_corr=min(1, p * factor), holds=bool(dirn and p * factor < 0.05)))
    return pd.DataFrame(rows)
PH = {}
for reading in "abc":
    for failures in "ab":
        PH[(reading, failures)] = posthoc(sel(G3, reading, failures), COMP, 3)
        print(f"M3 {reading}, M2 {failures}: {int(PH[(reading, failures)].holds.sum())} of 8 hold")
print("\nM3 a, M2 a:")
print(PH[("a", "a")].to_string(float_format=lambda x: f"{x:.3g}"))
best = max(PH, key=lambda k: PH[k].holds.sum())
print(f"\nbest reading M3 {best[0]}, M2 {best[1]}:")
print(PH[best].to_string(float_format=lambda x: f"{x:.3g}"))

**Result: not reproduced.** At most 4 of the 8 comparisons hold (M3 a, either M2): the four AAC against
CCK+BC differences in rise time, latency, amplitude and potency (corrected p 0.0112 to 0.0293). The PV+BC
against CCK+BC differences go the stated way but miss the corrected threshold (corrected p 0.205, 0.0512,
0.0663), and the AAC against CCK+BC transmission probability gives corrected p 0.159. The 1 Hz reading
gives none, the pooled reading 2.

## fig4c-cckdti-pc-silent-single-ap

> Seven of eight CCK+DTI to PC pairs transmit neither for a single AP nor for an AP after a RIP burst; a
> single AP evokes a uIPSC only after an EPI burst.

A pair transmits when any sweep succeeds (M2 a) or when the detected rate exceeds the rate in the same
window 100 ms before the AP (M2 b). Pairs that `Experiments_description.txt` marks "No release" for RIP
have no RIP recording and are counted as silent.

In [ ]:
# fig4c-cckdti-pc-silent-single-ap: transmission to the single AP before the RIP burst and to the
# single AP before the EPI burst (which follows the previous sweep's EPI train), CCK+DTI to PC pairs.
def tp_reading_b(path, col, shift=0.0):
    """Fraction of sweeps with a detected uIPSC (M2 b) after AP `col`; with shift < 0 the same window
    `shift` ms before the AP, where no AP occurs (the crossing rate with no AP)."""
    tr = trials(path, [col])
    if not tr:
        return np.nan
    tt, ys = hc7.snippets(RECS[path], [s for s, *_ in tr], [t + shift for _, t, *_ in tr])
    return hc7.detect_success(tt, ys).mean()
rows = []
for key in sorted(k for k in PAIRS if k[0] == "CCK+DTI to PC"):
    pr = PAIRS[key]
    dpair = fmap.get(key, ["?"])[0]
    lab = EX[(EX.connection == "CCK+DTI to PC") & (EX.pair == int(dpair))].set_index("protocol").entry.to_dict()
    row = dict(pair=key[1], desc_pair=dpair, RIP_label=lab.get("RIP", ""), EPI_label=lab.get("EPI", ""))
    for proto in ("RIP", "EPI"):
        if proto in pr and pr[proto] in EPA:
            e = EPA[pr[proto]]
            row[f"{proto}_tp_a"] = np.nanmean(e.success[:, 0])
            row[f"{proto}_tp_b"] = tp_reading_b(pr[proto], 0)
            row[f"{proto}_null_b"] = tp_reading_b(pr[proto], 0, shift=-100.0)
        else:
            row[f"{proto}_tp_a"] = row[f"{proto}_tp_b"] = row[f"{proto}_null_b"] = np.nan
    rows.append(row)
F4C = pd.DataFrame(rows)
F4C["RIP_single_transmits_a"] = F4C.RIP_tp_a > 0
F4C["RIP_single_transmits_b"] = F4C.RIP_tp_b > F4C.RIP_null_b
F4C["EPI_single_transmits_a"] = F4C.EPI_tp_a > 0
F4C["EPI_single_transmits_b"] = F4C.EPI_tp_b > F4C.EPI_null_b
print(F4C.round(2).to_string())
for r in "ab":
    silent = (~F4C[f"RIP_single_transmits_{r}"]).sum()
    print(f"M2 {r}: {silent} of {len(F4C)} pairs silent to the single AP before RIP "
          f"({silent / len(F4C):.0%}; pairs with no RIP recording, labelled 'No release', count as silent); "
          f"{int(F4C[f'EPI_single_transmits_{r}'].sum())} transmit to the single AP before EPI")
# per sweep, the authors' success (M2 a) at the single AP before RIP, in sweep order; a variant counts a pair as
# silent when it has no success in its first three sweeps (the paper's build-up of transmission over sweeps,
# claim cckdti-pc-transmission, "starting from the fourth trace"); not the claims file's criterion
print("\nsuccess at the single AP before RIP, per sweep in recording order (M2 a):")
early_silent = 0
for key in sorted(k for k in PAIRS if k[0] == "CCK+DTI to PC"):
    p = PAIRS[key].get("RIP")
    if p is None or p not in EPA:
        print(f"  {key[1]}: no RIP recording")
        early_silent += 1
        continue
    s = EPA[p].success[:, 0]
    first3 = np.nansum(s[:3])
    early_silent += int(first3 == 0)
    print(f"  {key[1]}: {s.astype(int).tolist()} | successes in sweeps 1 to 3: {int(first3)}, in later sweeps: {int(np.nansum(s[3:]))} of {len(s) - 3}")
print(f"variant, silent in the first three sweeps (or no RIP recording): {early_silent} of {len(F4C)} pairs ({early_silent / len(F4C):.0%})")
# the single AP of the first sweep is the only one not preceded by a RIP burst 1600 ms earlier (Methods, "1600 ms
# intervals between the last pulse of the trains and the subsequent single pulses")
first_silent = sum(1 for key in sorted(k for k in PAIRS if k[0] == "CCK+DTI to PC")
                   if PAIRS[key].get("RIP") not in EPA or EPA[PAIRS[key]["RIP"]].success[0, 0] == 0)
print(f"variant, silent in the first sweep, the only single AP without a preceding burst (or no RIP recording): "
      f"{first_silent} of {len(F4C)} pairs ({first_silent / len(F4C):.0%})")

**Result: not reproduced.** The data set holds 7 CCK+DTI to PC pairs. Three are silent to the single AP
before RIP (all three marked "No release", 43%, against the 85% the criterion requires); the four with a
RIP recording transmit to that AP with success rates 0.55 to 0.85 (M2 a) and detection rates above the
no-AP rate (M2 b). All seven transmit to the single AP that follows an EPI train under M2 a (six under
M2 b). The paper itself says elsewhere that four of eight pairs respond at 1 to 40 Hz or to RIP (claim
cckdti-pc-transmission), which agrees better with these data than "seven of eight".

Transmission builds up over the sweeps: none of the four pairs succeeds in the first sweep, and two
succeed once in sweeps 1 to 3. Counting a pair as silent when it fails in the first three sweeps gives 5
of 7 (71%), still below 85%; counting only the first sweep, the only single AP not preceded by a RIP burst
1600 ms earlier, gives 7 of 7. The criterion counts any success in any sweep, so the outcome stands; the
reading of the first sweeps is asked of a reader (`questions/crcns-hc-7.yaml`).

## Interneuron pairs: fig8-pvbc-pvbc-uipsc, fig8-cck-pairs-uipsc, fig8-cckbc-vs-cckdti-pairs

> uIPSCs between PV+BCs (n = 16) are fast (10 to 90% rise time 0.63 ms, decay time constant 3.17 ms),
> with short latency (0.76 ms), large amplitude (81.18 pA) and transmission probability of 1.

> uIPSCs between CCK+BCs (n = 6) have rise and decay times like those between PV+BCs (0.63 and 3.65 ms)
> but significantly longer latency (1.59 ms) and lower transmission probability (0.7); uIPSCs between
> CCK+DTIs (n = 4) are slowest (rise 0.98 ms, decay 4.47 ms, latency 1.52 ms), smallest (43.71 pA) and
> least reliable (0.55).

> Among CCK+ pairs, uIPSC decay, latency, amplitude and transmission probability are similar between
> CCK+BC and CCK+DTI pairs, but CCK+BC uIPSCs rise significantly faster.

No interneuron pair has a RIP recording (M3 a), and the CCK+ pairs have no 1 Hz train (M3 b, c): they were
recorded only with 10, 40 and 160 Hz trains. None of the claims file's readings can be run on them, so
fig8-cck-pairs-uipsc and fig8-cckbc-vs-cckdti-pairs are blocked (`data_missing`). Reading d, the first AP
of each train at 40 Hz or slower, which the claims file does not list, is shown for them and does not
decide. The PV+BC pairs are shown under M3 b, c and d.

In [ ]:
# fig8-pvbc-pvbc-uipsc, fig8-cck-pairs-uipsc, fig8-cckbc-vs-cckdti-pairs: interneuron pairs.
# PV+BC pairs have the 1 Hz train (M3 b, c); no pair has a RIP recording (M3 a); the CCK+ pairs have neither,
# so their values come from reading d (first APs of trains), which the claims file does not list.
G8 = ["PV+BC to PV+BC", "CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI"]
EXP8 = {"PV+BC to PV+BC": dict(rise=0.63, tau=3.17, latency=0.76, amplitude=81.18, tp=1.0),
        "CCK+BC to CCK+BC": dict(rise=0.63, tau=3.65, latency=1.59, tp=0.7),
        "CCK+DTI to CCK+DTI": dict(rise=0.98, tau=4.47, latency=1.52, amplitude=43.71, tp=0.55)}
def within(v, e):
    return abs(v - e) <= 0.1 * abs(e) + 1e-9
F8 = []
for reading in "bcd":
    for failures in "ab":
        for dw in hc7.DECAY_WINS:
            d = sel(G8, reading, failures, dw)
            for g in G8:
                dd = d[d.type == g]
                for m, e in EXP8[g].items():
                    v = dd[m].median()
                    F8.append(dict(M3=reading, M2=failures, decay_win=dw, type=g, n=len(dd), measure=m, expected=e,
                                   median=v, within=bool(np.isfinite(v) and within(v, e))))
F8 = pd.DataFrame(F8)
print("PV+BC to PV+BC pairs with a 1 Hz train (M3 b):", sorted(k[1] for k in PAIRS if k[0] == "PV+BC to PV+BC" and "1 Hz" in PAIRS[k]),
      "of", sorted(k[1] for k in PAIRS if k[0] == "PV+BC to PV+BC"))
print("CCK+ to CCK+ pairs with RIP or a 1 Hz train (M3 a to c):",
      [k for k in PAIRS if k[0] in G8[1:] and ("RIP" in PAIRS[k] or "1 Hz" in PAIRS[k])])
print("medians over pairs against the expected values (tolerance 10%):")
print(F8[F8.decay_win == 20].pivot_table(index=["type", "measure", "expected"], columns=["M3", "M2"], values="median").round(3).to_string())
print("\nnumber within tolerance, per reading:")
print(F8.groupby(["M3", "M2", "decay_win", "type"]).within.sum().unstack().to_string())
print("\nranges over pairs (M3 d, M2 a, 20 ms):")
print(sel(G8, "d", "a").groupby("type")[MEAS].agg(["min", "max"]).round(2).to_string())
# tests across the three pair types (reading d is the only one that covers all three)
for failures in "ab":
    d = sel(G8, "d", failures)
    print(f"\nM3 d, M2 {failures}: Kruskal-Wallis p across the three pair types:",
          {m: round(stats.kruskal(*[d[d.type == g][m].dropna() for g in G8]).pvalue, 4) for m in MEAS})
    comps8 = [("latency", "CCK+BC to CCK+BC", "PV+BC to PV+BC", ">"), ("tp", "CCK+BC to CCK+BC", "PV+BC to PV+BC", "<")]
    print(posthoc(d, comps8, 3).to_string(float_format=lambda x: f"{x:.3g}"))
    comps_cc = [("rise", "CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI", "<")] + \
               [(m, "CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI", "=") for m in ["tau", "latency", "amplitude", "tp"]]
    ph = posthoc(d, comps_cc, 3)
    ph.loc[ph.expected == "=", "holds"] = ph.p_corr >= 0.05
    print(ph.to_string(float_format=lambda x: f"{x:.3g}"))

**Result: fig8-pvbc-pvbc-uipsc partially reproduced.** Over 10 PV+BC pairs with the 1 Hz train (M3 b),
3 of the 5 medians are within 10%: decay 3.38 ms (3.17), latency 0.687 ms (0.76; tolerance 0.076) and
transmission probability 1.0. The rise time is faster (0.514 against 0.63 ms) and the amplitude larger
(104.4 against 81.18 pA). Reading d (first AP of the trains, 14 pairs) gives 2 of 5. The data set holds
14 of the paper's 16 pairs, and 10 of them have the 1 Hz train.

**Result: fig8-cck-pairs-uipsc blocked (data_missing).** The CCK+ pairs have no RIP recording and no 1 Hz
train, so none of the claims file's single-AP readings exists. Under reading d, shown only, at most 2 of
the 9 CCK+ medians are within tolerance (1 under M2 a, 2 under M2 b), CCK+BC pairs have a lower transmission
probability than PV+BC pairs (corrected p 0.00164) and not a longer latency (corrected p 1). Reading d is
itself doubtful here: one CCK+BC pair has a negative latency (-0.12 ms) and the CCK+DTI amplitudes are 2.7 to
14.2 pA.

**Result: fig8-cckbc-vs-cckdti-pairs blocked (data_missing).** As fig8-cck-pairs-uipsc. Under reading d,
shown only, CCK+BC pairs rise more slowly than CCK+DTI pairs (1.43 against 0.82 ms, corrected p 0.514, M2
a), the opposite of the claim.

## decay-pc-vs-interneuron

> uIPSCs from PV+BCs and CCK+BCs decay more slowly in PCs than in homologous interneurons (PV+BC, 7.31
> against 3.17 ms; CCK+BC, 6.34 against 4.47 ms).

PC side by M3 a to c; interneuron side, PV+BC pairs by M3 b or d and CCK+ pairs by reading d; the CCK+
comparison group by the three readings of the criterion; decay fits over 20 and 50 ms.

In [ ]:
# decay-pc-vs-interneuron: decay of uIPSCs on PCs against homologous interneuron pairs.
rows = []
for dw in hc7.DECAY_WINS:
    for failures in "ab":
        for r_pc in "abc":
            for r_in in ("b/d", "d"):   # interneuron pairs: PV+BC pairs by M3 b (1 Hz) or d, CCK+ pairs by d
                pc = sel(["PV+BC to PC", "CCK+BC to PC"], r_pc, failures, dw)
                inn = pd.concat([sel(["PV+BC to PV+BC"], r_in[0], failures, dw),
                                 sel(["CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI"], "d", failures, dw)])
                for cck_in in ("a: CCK+BC pairs", "b: CCK+DTI pairs", "c: both"):
                    cin = {"a": ["CCK+BC to CCK+BC"], "b": ["CCK+DTI to CCK+DTI"], "c": ["CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI"]}[cck_in[0]]
                    x1, y1 = pc[pc.type == "PV+BC to PC"].tau.dropna(), inn[inn.type == "PV+BC to PV+BC"].tau.dropna()
                    x2, y2 = pc[pc.type == "CCK+BC to PC"].tau.dropna(), inn[inn.type.isin(cin)].tau.dropna()
                    if min(len(x1), len(y1), len(x2), len(y2)) == 0:
                        continue
                    p1 = stats.mannwhitneyu(x1, y1).pvalue; p2 = stats.mannwhitneyu(x2, y2).pvalue
                    meds = [x1.median(), y1.median(), x2.median(), y2.median()]
                    nw = sum(within(v, e) for v, e in zip(meds, [7.31, 3.17, 6.34, 4.47]))
                    rows.append(dict(decay_win=dw, M2=failures, M3_pc=r_pc, M3_in=r_in, cck_group=cck_in, PVPC=meds[0], PVPV=meds[1],
                                     CCKPC=meds[2], CCKIN=meds[3], n=f"{len(x1)}/{len(y1)}/{len(x2)}/{len(y2)}", p_PV=p1, p_CCK=p2,
                                     n_within=nw, holds=bool(p1 < 0.05 and p2 < 0.05 and meds[0] > meds[1] and meds[2] > meds[3] and nw >= 3)))
DEC = pd.DataFrame(rows)
print(DEC.to_string(float_format=lambda x: f"{x:.3g}"))
print("\nreadings under which the claim holds:", int(DEC.holds.sum()), "of", len(DEC))

**Result: blocked (data_missing).** The claim depends on fig8-cck-pairs-uipsc, blocked: the CCK+BC to
CCK+BC decay can only be measured by reading d, which the claims file does not list. The PV+BC part holds
under every reading: PV+BC to PC 7.52 to 8.07 ms against PV+BC to PV+BC 3.38 to 3.99 ms, p 0.000206 to
0.002, both medians within 10% under the first reading (7.78 against 7.31; 3.38 against 3.17 ms). Under
reading d, shown only, the CCK+BC part fails: CCK+BC to PC 6.12 to 7.04 ms against 4.90 to 8.09 ms in
the CCK+ interneuron groups, p 0.0698 or more.

## Amplitudes in bursts and trains

In [ ]:
# Mean IPSC amplitude at each AP, per pair and protocol (claims M4, M5). Reading a: the per-sweep
# amplitudes of the _EPA table (0 for failures), averaged over sweeps; reading b: the peak of the
# sweep-averaged raw trace between each AP and the next (hc7.local_amplitudes, applied to the sweep average). M5: all sweeps, and a
# variant without the first three sweeps. Relative amplitude: each AP over the first AP of the same
# train (for RIP and EPI, the first AP of the burst, after the single AP; claims M4).
def amps(key, proto, reading="a", drop=0):
    path = PAIRS[key].get(proto)
    if path is None or path not in EPA:
        return None
    e, rec = EPA[path], RECS[path]
    if reading == "a":
        a = e.amp[drop:]
        out = np.nanmean(a, 0)
        if e.ar_first is not None:   # columns marked "AR" in the table's mean row
            out[e.ar_first:] = np.nan
        return out
    sweeps = [sw - 1 for sw in e.sweeps[drop:] if sw - 1 < len(rec.post)]
    times, present = hc7.stim_template(rec, proto)
    sweeps = [s for s in sweeps if present[s].all()]
    if not sweeps:
        return None
    r = rec.rate / 1e3
    avg = -np.mean(rec.post[sweeps].astype(float), 0)
    out = []
    for j, ta in enumerate(times):
        tn = times[j + 1] if j + 1 < len(times) else ta + 10.0
        b = avg[int(round((ta - 1.3) * r)):int(round((ta - 0.8) * r))].mean()
        seg = avg[int(round((ta + 0.3) * r)):int(round(tn * r)) + 1]
        out.append(seg.max() - b)
    return np.array(out)

def rel(key, proto, reading="a", drop=0):
    a = amps(key, proto, reading, drop)
    if a is None:
        return None
    first = 1 if proto in ("RIP", "EPI") else 0
    return a / a[first] if a[first] > 0 else np.full(len(a), np.nan)
print("example, PV+BC to PC Pair02, 10 Hz, readings a and b (pA):")
print(np.round(amps(("PV+BC to PC", "Pair02"), "10 Hz", "a"), 1))
print(np.round(amps(("PV+BC to PC", "Pair02"), "10 Hz", "b"), 1))

## fig5c-pv-rip-depression and fig5c-cck-rip-facilitation

> Under the RIP protocol (four APs at 160 Hz) PV+BC to PC pairs show short-term depression at the 3rd and
> 4th AP, and AAC to PC pairs already at the 2nd AP.

> CCK+BC to PC (n = 14) and CCK+DTI to PC (n = 4) pairs on average show strong short-term facilitation,
> although individual pairs can show either facilitation or depression.

The RIP bursts in the data are at 222 Hz (data check).

In [ ]:
# fig5c-pv-rip-depression and fig5c-cck-rip-facilitation: relative amplitudes of the four RIP IPSCs.
def rip_table(types, reading, drop):
    rows = []
    for key in sorted(k for k in PAIRS if k[0] in types):
        r = rel(key, "RIP", reading, drop)
        if r is not None:
            rows.append(dict(type=key[0], pair=key[1], ap2=r[2], ap3=r[3], ap4=r[4]))
    return pd.DataFrame(rows)
def test_vs_1(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    w = stats.wilcoxon(x - 1).pvalue if len(x) > 1 and np.any(x != 1) else np.nan
    u = stats.mannwhitneyu(x, np.ones(len(x)), alternative="two-sided").pvalue if len(x) else np.nan
    return np.median(x) if len(x) else np.nan, w, u
RIPR = {}
for reading in "ab":
    for drop in (0, 3):
        t = rip_table(IN_PC, reading, drop)
        RIPR[(reading, drop)] = t
        rows = []
        for g in IN_PC:
            for ap in ("ap2", "ap3", "ap4"):
                med, w, u = test_vs_1(t[t.type == g][ap])
                rows.append(dict(type=g, ap=ap, n=int((t.type == g).sum()), median=med, wilcoxon_p=w, mwu_p=u))
        R = pd.DataFrame(rows)
        print(f"\nM4 {reading}, first {drop} sweeps left out:")
        print(R.to_string(float_format=lambda x: f"{x:.3g}"))
        for test in ("wilcoxon_p", "mwu_p"):
            q = R.set_index(["type", "ap"])
            ok = all(q.loc[(g, a), "median"] < 1 and q.loc[(g, a), test] < 0.05
                     for g, a in [("PV+BC to PC", "ap3"), ("PV+BC to PC", "ap4"), ("AAC to PC", "ap2")])
            cck = q.loc["CCK+BC to PC"]["median"]
            dti = q.loc["CCK+DTI to PC"]["median"]
            t4 = t[t.type.isin(["CCK+BC to PC", "CCK+DTI to PC"])].ap4
            print(f"  test {test}: PV+BC 3rd and 4th and AAC 2nd depressed: {ok}; PV+BC 2nd: median "
                  f"{q.loc[('PV+BC to PC', 'ap2'), 'median']:.3f}, p {q.loc[('PV+BC to PC', 'ap2'), test]:.3g}")
        print(f"  CCK+BC medians above 1 at {int((cck > 1).sum())} of 3 APs; CCK+DTI at {int((dti > 1).sum())} of 3; "
              f"CCK+ pairs below 1 at the 4th AP: {int((t4 < 1).sum())} of {t4.notna().sum()}")
print("\nper pair, M4 a, all sweeps:")
print(RIPR[("a", 0)].round(3).to_string())

**Result: fig5c-pv-rip-depression reproduced.** With the raw-trace amplitudes (M4 b) and the Wilcoxon
test (reading a), the PV+BC medians at the 3rd and 4th AP (0.372, 0.329; p 0.0156) and the AAC median at
the 2nd AP (0.679, p 0.0312) are below 1. With the authors' amplitudes (M4 a) it holds only under the
Mann-Whitney reading (PV+BC 0.650 and 0.641, p 0.00106; AAC 0.767, p 0.0204); with the Wilcoxon test the
AAC 2nd AP misses (p 0.109). The Mann-Whitney reading compares the values with a constant vector of ones
and gives p 0.00106 whenever all 7 values lie on one side of 1, so it is a weak test; this is asked of a
reader. The PV+BC 2nd AP is not depressed (median 1.15 under M4 a, 0.969 under M4 b, p 0.172 or more).

**Result: fig5c-cck-rip-facilitation reproduced.** Under M4 a the CCK+BC medians are above 1 at the 2nd to
4th AP (1.37, 1.46, 1.11; 12 pairs) and so are those of the 4 CCK+DTI pairs that transmit under RIP;
7 of the 16 CCK+ pairs are below 1 at the 4th AP. Under M4 b the facilitation is weaker (CCK+BC above 1 at
1 of 3 APs), as local amplitudes at 222 Hz are measured on the decay of the previous IPSC.

## fig5db-recovery

> After a RIP train, PV+BC to PC (n = 7) and AAC to PC (n = 9) transmission recovers gradually with the
> interval to a second RIP train, while CCK+BC to PC transmission (n = 11) rises and falls with the
> interval; PV+BC and CCK+BC pairs differ at several intervals.

In [ ]:
# fig5db-recovery and fig13-pvbc-pc-recovery-halftime: recovery ratio, the first IPSC of the second RIP
# burst over the first IPSC of the first, per pair and interval. The interval of each sweep is read from
# its AP times (the gap from the last AP of the first burst to the first AP of the second, nominal values
# 50 to 1200 ms); the _EPA table groups sweeps k, k+7, k+14, ... in its k-th block.
def recovery(key, reading="a"):
    path = PAIRS[key].get("Recovery")
    if path is None:
        return None
    rec = RECS[path]
    iv = np.array([v for _, v in hc7.recovery_sweeps(rec)])
    out = {}
    if reading == "a":
        if path not in EPA:
            return None
        e = EPA[path]
        for b in range(e.mean_amp.shape[0]):
            sw = e.sweeps[e.block == b] - 1
            ivs = set(iv[sw[sw < len(iv)]][np.isfinite(iv[sw[sw < len(iv)]])])
            if len(ivs) != 1:
                continue
            a = np.nanmean(e.amp[e.block == b][:, :8], 0)
            out[ivs.pop()] = a[4] / a[0] if a[0] > 0 else np.nan
    else:
        r = rec.rate / 1e3
        for v in hc7.RECOVERY_INTERVALS:
            sw = np.flatnonzero(iv == v)
            if len(sw) == 0:
                continue
            avg = -np.mean(rec.post[sw].astype(float), 0)
            t = np.median(np.array([rec.ap[s][:8] for s in sw]), 0)
            pk = []
            for j in (0, 4):
                b = avg[int(round((t[j] - 1.3) * r)):int(round((t[j] - 0.8) * r))].mean()
                pk.append(avg[int(round((t[j] + 0.3) * r)):int(round((t[j] + 4.0) * r))].max() - b)
            out[v] = pk[1] / pk[0]
    return out
print("block intervals in recording order, PV+BC to PC Pair02:",
      [v for _, v in hc7.recovery_sweeps(RECS[PAIRS[("PV+BC to PC", "Pair02")]["Recovery"]])][:7])
REC = {}
for reading in "ab":
    rows = []
    for key in sorted(k for k in PAIRS if k[0] in IN_PC):
        r = recovery(key, reading)
        if r:
            rows += [dict(type=key[0], pair=key[1], interval=k, ratio=v) for k, v in r.items()]
    REC[reading] = pd.DataFrame(rows)
for reading in "ab":
    R = REC[reading]
    print(f"\nM4 {reading}: median recovery ratio by type and interval (pairs per type: "
          f"{R.groupby('type').pair.nunique().to_dict()})")
    med = R.pivot_table(index="type", columns="interval", values="ratio", aggfunc="median")
    print(med.round(3).to_string())
    for g in ("PV+BC to PC", "AAC to PC"):
        d = R[R.type == g]
        rho, p = stats.spearmanr(d.interval, d.ratio)
        print(f"  {g}: Spearman rho {rho:.3f}, p {p:.3g}; median at 50 ms {med.loc[g, 50]:.3f}")
    m = med.loc["CCK+BC to PC"].values
    rises_then_falls = any(m[i] > m[:i].min() and m[j] < 0.9 * m[i] for i in range(1, len(m)) for j in range(i + 1, len(m)))
    print("  CCK+BC median ratios:", np.round(m, 3), "| a rise followed by a fall of more than 10%:", rises_then_falls)
    rows = []
    for v in hc7.RECOVERY_INTERVALS:
        x = R[(R.type == "PV+BC to PC") & (R.interval == v)].ratio
        y = R[(R.type == "CCK+BC to PC") & (R.interval == v)].ratio
        p = stats.mannwhitneyu(x, y).pvalue
        rows.append(dict(interval=v, n_pv=len(x), n_cck=len(y), p=p, p_bonf7=min(1, 7 * p)))
    print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.3g}"))

**Result: reproduced.** Under M4 a (6 PV+BC, 7 AAC, 9 CCK+BC pairs) the ratio rises with the interval for
PV+BC (Spearman rho 0.306, p 0.0484) and AAC (rho 0.527, p 9.85e-05), both below 1 at 50 ms (0.634); the
CCK+BC median rises from 0.776 to 1.052 at 500 ms and falls to 0.725 at 1200 ms. PV+BC and CCK+BC differ
at 100 ms (p 0.012) and 200 ms (p 0.0496) uncorrected; with Bonferroni over 7 intervals no interval is
significant (smallest 0.0839). Under M4 b the same holds (PV+BC rho 0.385, p 0.00626; 100 ms p 0.0115).
The result rests on the uncorrected reading.

## fig13-pvbc-pc-recovery-halftime

> Transmission at PV+BC to PC synapses recovers after a RIP train with a half-time of 495.8 ms
> (interquartile range 343.6 to 648.0, n = 7), close to the recovery of SWR amplitude.

In [ ]:
# fig13-pvbc-pc-recovery-halftime: per PV+BC to PC pair, an exponential recovery fitted to the ratios.
from scipy.optimize import curve_fit
def fit_recovery(t, r, free_plateau):
    try:
        if free_plateau:
            f = lambda t, r0, rmax, tau: rmax - (rmax - r0) * np.exp(-t / tau)
            p, _ = curve_fit(f, t, r, p0=(0.5, 1.0, 400), bounds=([-2, 0, 1], [2, 5, 1e5]), maxfev=20000)
            return p[2]
        f = lambda t, r0, tau: 1 - (1 - r0) * np.exp(-t / tau)
        p, _ = curve_fit(f, t, r, p0=(0.5, 400), bounds=([-2, 1], [2, 1e5]), maxfev=20000)
        return p[1]
    except RuntimeError:
        return np.nan
rows = []
for reading in "ab":
    R = REC[reading]
    for pair, d in R[R.type == "PV+BC to PC"].groupby("pair"):
        d = d.sort_values("interval")
        for fp in (False, True):
            tau = fit_recovery(d.interval.values, d.ratio.values, fp)
            rows.append(dict(M4=reading, pair=pair, free_plateau=fp, tau=tau, half_time=tau * np.log(2)))
HT = pd.DataFrame(rows)
print(HT.pivot_table(index=["M4", "pair"], columns="free_plateau", values="tau").round(1).to_string())
print("fits at the tau bound of 1e5 ms, by reading and fit:", HT.assign(at_bound=HT.tau >= 0.99e5)
      .groupby(["M4", "free_plateau"]).at_bound.sum().astype(int).to_dict())
summ = []
for (reading, fp), d in HT.groupby(["M4", "free_plateau"]):
    for val in ("tau", "half_time"):
        v = d[val].median()
        summ.append(dict(M4=reading, fit="free plateau (c, d)" if fp else "plateau 1 (a, b)", value=val, n=d[val].notna().sum(),
                         median=v, q1=d[val].quantile(.25), q3=d[val].quantile(.75), within=abs(v - 495.8) <= 152.2))
print(pd.DataFrame(summ).round(1).to_string())

**Result: reproduced.** With a free plateau the median tau is 525.8 ms (M4 a, 6 pairs) and 433.6 ms (M4 b,
7 pairs), both within 495.8 plus or minus 152.2 ms, and tau ln 2 under M4 a is 364.4 ms, also within.
With the plateau fixed at 1 the medians are 1323.6 to 1538.3 ms (outside). The single fits are poorly
constrained: the interquartile ranges reach thousands of ms (295.0 to 5353.5 ms for the free plateau
under M4 a, against the paper's 343.6 to 648), and up to 2 fits per reading reach the bound of 100000 ms
(2 with the plateau at 1 under M4 a; 1 of each fit under M4 b; none with the free plateau under M4 a).
Only 3 of the 8 combinations of reading and measure land within tolerance, so the agreement of the
median is fragile.

## Trains: fig6-pv-trains-depression and fig9-pvbc-pvbc-depression

> PV+BC to PC (n = 6) and AAC to PC (n = 8) pairs show purely short-term depression to 10 APs at 1 to 320
> Hz and to the EPI protocol, more pronounced at higher frequencies, with IPSCs synchronous to the APs
> even under EPI.

> Transmission between PV+BCs (n = 13) depresses during 10-AP trains at all frequencies from 1 to 320 Hz,
> more strongly at higher frequencies.

In [ ]:
# Relative amplitudes in the 10-AP trains and the EPI burst (claims M4, M5), per pair.
TRAIN_PROTOS = [f"{f} Hz" for f in hc7.FREQS]
def train_rel(types, reading="a", drop=0):
    rows = []
    for key in sorted(k for k in PAIRS if k[0] in types):
        for proto in TRAIN_PROTOS + ["EPI"]:
            r = rel(key, proto, reading, drop)
            if r is None:
                continue
            first = 1 if proto == "EPI" else 0
            rr = r[first:]
            if not np.isfinite(rr).any():   # first IPSC of zero mean amplitude: cannot be normalized
                continue
            rows.append(dict(type=key[0], pair=key[1], protocol=proto, n_ap=len(rr), mean_2_10=np.nanmean(rr[1:10]),
                             max_2_10=np.nanmax(rr[1:10]), rel10=rr[9] if len(rr) > 9 else np.nan,
                             rel11=rr[10] if len(rr) > 10 and proto != "EPI" else np.nan,
                             # the last AP of the train: the 10th of a 10-AP train (the 11th comes 1 s later), the 34th
                             # of the EPI burst (NaN under M4 a where the table marks it "AR")
                             last=rr[9] if proto != "EPI" else rr[-1],
                             last_finite=rr[np.flatnonzero(np.isfinite(rr))[-1]], peak_2_5=np.nanmax(rr[1:5]), rel=rr))
    return pd.DataFrame(rows)
TR = {(reading, drop): train_rel(IN_PC + IN_IN, reading, drop) for reading in "ab" for drop in (0, 3)}
def freq(p):
    return int(p.split()[0]) if p.endswith("Hz") else np.nan
for k in TR:
    TR[k]["freq"] = TR[k].protocol.map(freq)
print("pairs with each train protocol (M4 a):")
print(TR[("a", 0)].pivot_table(index="type", columns="protocol", values="pair", aggfunc="count").to_string())

# fig6-pv-trains-depression (depression parts) and fig9-pvbc-pvbc-depression
def depression_parts(t, g):
    d = t[t.type == g]
    med = d.groupby("protocol").rel.apply(lambda s: np.nanmedian(np.vstack([x[:10] for x in s]), 0))
    above = {p: int((m[1:10] > 1).sum()) for p, m in med.items()}
    maxv = {p: float(np.nanmax(m[1:10])) for p, m in med.items()}
    pure = all(v <= 1.05 and above[p] <= 1 for p, v in maxv.items())
    dd = d[d.protocol != "EPI"]
    rho, pr = stats.spearmanr(dd.freq, dd.mean_2_10)
    m10 = d.groupby("protocol").rel10.median()
    return dict(pure=pure, max_median_2_10=maxv, aps_above_1=above, rho=rho, p=pr, rel10_median=m10.round(3).to_dict(),
                median_mean_2_10=dd.groupby("freq").mean_2_10.median().round(3).to_dict())
for k, t in TR.items():
    print(f"\nM4 {k[0]}, first {k[1]} sweeps left out")
    for g in ("PV+BC to PC", "AAC to PC", "PV+BC to PV+BC"):
        r = depression_parts(t, g)
        print(f"  {g}: purely depressing {r['pure']} (largest median relative amplitude of APs 2 to 10 per protocol "
              f"{ {p: round(v, 3) for p, v in r['max_median_2_10'].items()} }); Spearman of mean APs 2 to 10 with frequency "
              f"rho {r['rho']:.3f}, p {r['p']:.3g}")
        print(f"     median mean of APs 2 to 10 by frequency: {r['median_mean_2_10']}")
        if g == "PV+BC to PV+BC":
            print(f"     median 10th AP by protocol: {r['rel10_median']}")

for k in [("a", 0), ("b", 0)]:
    t = TR[k]
    for g in ("PV+BC to PC", "AAC to PC"):
        d = t[(t.type == g) & (t.protocol == "1 Hz")]
        print(f"M4 {k[0]}, {g}, 1 Hz ({len(d)} pairs): median relative amplitude at APs 1 to 10",
              np.round(np.nanmedian(np.vstack([x[:10] for x in d.rel]), 0), 3))

**Result: fig6-pv-trains-depression not reproduced.** The depression grows with frequency for both types
(Spearman rho -0.738 for PV+BC, -0.788 for AAC under M4 a; p 1.9e-07 and 1.1e-11), and every protocol
from 5 Hz up and EPI has all medians of APs 2 to 10 below 1. At 1 Hz (4 PV+BC and 3 AAC pairs) the medians
fluctuate around 1: under M4 a they reach 1.148 for PV+BC and 1.156 for AAC, above the 1.05 allowed;
under M4 b the PV+BC median reaches 1.128 and the AAC medians exceed 1 at two APs (1.006 and 1.003, where
one is allowed). "Purely depressing at every frequency" therefore fails under every reading. The synchrony part fails too: under EPI onset rule (a) flags asynchronous charge
in 6 of 7 PV+BC and 8 of 8 AAC pairs and rule (b) in 7 of 7 and 8 of 8 (section "Asynchronous release").

**Result: fig9-pvbc-pvbc-depression reproduced.** Over 10 to 14 PV+BC pairs per frequency the median
relative amplitude at the 10th AP is below 1 at all eight frequencies (0.806 at 1 Hz to 0.14 at 320 Hz,
M4 a), and the mean of APs 2 to 10 falls with frequency (rho -0.737, p 2.4e-18). M4 b and the variant
without the first three sweeps agree.

## fig9-pvbc-pvbc-transmission-probability and fig9-pvbc-pvbc-recovery-11th

> Between PV+BCs the transmission probability starts to fall at 80 Hz (to 0.9 at the 10th AP), falls by 5
> to 10% at the 8th to 10th AP at 160 Hz, and lies between 0.8 and 0.9 from the 3rd to the 10th AP at
> 320 Hz; it recovers by the 11th AP.

> Between PV+BCs a 1 s pause does not fully relieve depression, and the recovery of the 11th IPSC does not
> depend on the frequency of the preceding train.

In [ ]:
# fig9-pvbc-pvbc-transmission-probability: per pair and AP the fraction of sweeps with a uIPSC (M2 a, the
# _EPA success table), median over PV+BC to PV+BC pairs.
tp_rows = []
for key in sorted(k for k in PAIRS if k[0] == "PV+BC to PV+BC"):
    for proto in TRAIN_PROTOS:
        p = PAIRS[key].get(proto)
        if p in EPA:
            tp = np.nanmean(EPA[p].success, 0)
            tp_rows.append(dict(pair=key[1], protocol=proto, tp=tp))
TP9 = pd.DataFrame(tp_rows)
med_tp = TP9.groupby("protocol").tp.apply(lambda s: np.nanmedian(np.vstack([np.pad(x, (0, 11 - len(x)), constant_values=np.nan) for x in s]), 0))
print("median transmission probability over PV+BC to PV+BC pairs, APs 1 to 11:")
for proto in TRAIN_PROTOS:
    print(f"  {proto:>6} (n={int((TP9.protocol == proto).sum())}):", np.round(med_tp[proto], 3))
tp80 = med_tp["80 Hz"][9]
dec160 = 100 * (1 - med_tp["160 Hz"][7:10] / med_tp["160 Hz"][0])
in320 = med_tp["320 Hz"][2:10]
below80 = {p: bool(np.all(med_tp[p][:10] == 1)) for p in TRAIN_PROTOS[:4]}
rec11 = {p: med_tp[p][10] for p in TRAIN_PROTOS if p != "1 Hz"}
print(f"80 Hz, 10th AP: {tp80:.3f} (expected 0.9 +- 0.09)")
print(f"160 Hz, decrease at the 8th to 10th AP from the 1st: {np.round(dec160, 1)} % (expected 4.5 to 11)")
print(f"320 Hz, 3rd to 10th AP: {np.round(in320, 3)} (expected 0.72 to 0.99); all inside: {bool(np.all((in320 >= 0.72) & (in320 <= 0.99)))}")
print("median 1 at every AP below 80 Hz:", below80)
print("11th AP median (at least 0.95):", {k: round(v, 3) for k, v in rec11.items()})

# fig9-pvbc-pvbc-recovery-11th: the 11th IPSC over the first, trains at 5 to 320 Hz.
for k in [("a", 0), ("b", 0)]:
    t = TR[k]
    d = t[(t.type == "PV+BC to PV+BC") & t.freq.between(5, 320)]
    med11 = d.groupby("freq").rel11.median()
    groups = [g.rel11.dropna().values for _, g in d.groupby("freq")]
    kw = stats.kruskal(*groups).pvalue
    wide = d.pivot_table(index="pair", columns="freq", values="rel11").dropna()
    fr = stats.friedmanchisquare(*[wide[c] for c in wide.columns]).pvalue if len(wide) > 2 else np.nan
    print(f"\nM4 {k[0]}: median 11th/1st by frequency {med11.round(3).to_dict()}; all below 1: {bool((med11 < 1).all())}; "
          f"Kruskal-Wallis p {kw:.3g}; Friedman p {fr:.3g} over {len(wide)} pairs with all seven frequencies")

# M2 reading b for the trains: per sweep and AP, the peak after the AP (to the next AP, at most 8 ms) above the
# current just before it, against 3 SD of the 10 ms before the first AP of the sweep (standard choice).
def tp_detect(path, proto):
    rec, e = RECS[path], EPA[path]
    times, present = hc7.stim_template(rec, proto)
    r = rec.rate / 1e3
    out = []
    for sw in e.sweeps:
        s = sw - 1
        if s >= len(rec.post):
            continue
        y = -rec.post[s].astype(float)
        i0 = int(round(times[0] * r))
        sd = y[i0 - int(10 * r):i0 - int(0.8 * r)].std()
        row = []
        for j, ta in enumerate(times):
            tn = min(times[j + 1] if j + 1 < len(times) else ta + 8, ta + 8)
            b = y[int(round((ta - 1.3) * r)):int(round((ta - 0.8) * r))].mean()
            row.append(float(y[int(round((ta + 0.3) * r)):int(round(tn * r)) + 1].max() - b > 3 * sd))
        out.append(row)
    return np.nanmean(np.array(out), 0)
tpb = collections.defaultdict(list)
for key in sorted(k for k in PAIRS if k[0] == "PV+BC to PV+BC"):
    for proto in ["80 Hz", "160 Hz", "320 Hz"]:
        p = PAIRS[key].get(proto)
        if p in EPA:
            tpb[proto].append(tp_detect(p, proto))
medb = {p: np.median(np.vstack(v), 0) for p, v in tpb.items()}
print("\nM2 b, median transmission probability (80, 160, 320 Hz):")
for p, m in medb.items():
    print(f"  {p}:", np.round(m, 3))
in320b = medb["320 Hz"][2:10]
print(f"M2 b: 80 Hz 10th AP {medb['80 Hz'][9]:.3f}; 160 Hz decrease 8th to 10th "
      f"{np.round(100 * (1 - medb['160 Hz'][7:10] / medb['160 Hz'][0]), 1)} %; 320 Hz 3rd to 10th inside 0.72 to 0.99: "
      f"{bool(np.all((in320b >= 0.72) & (in320b <= 0.99)))}")

**Result: fig9-pvbc-pvbc-transmission-probability not reproduced.** With the authors' success tables (M2
a) the median transmission probability at 80 Hz stays 1 at every AP (expected 0.9 plus or minus 0.09 at
the 10th); at 160 Hz it falls by 5, 10 and 5% at the 8th to 10th AP, as stated; at 320 Hz it is 1.0 at
the 3rd AP and 0.8 to 0.9 from the 4th to the 10th, so the 3rd AP is outside 0.72 to 0.99. It is 1 at
every AP below 80 Hz and at every 11th AP. Detection on the raw traces (M2 b) gives 1 at 80 Hz and 0.1 to
0.6 over the 3rd to 10th AP at 320 Hz, also outside. Neither the 80 Hz nor the 320 Hz part holds.

**Result: fig9-pvbc-pvbc-recovery-11th reproduced.** The median 11th over first IPSC is 0.61 to 0.71 at
every frequency from 5 to 320 Hz (M4 a), below 1, and does not depend on the frequency (Kruskal-Wallis p
0.124; Friedman over the 10 pairs with all seven frequencies p 0.115). M4 b gives p 0.668.

## CCK+ trains: fig6-cckbc-pc-trains, fig10-cck-cck-frequency, cckdti-pc-transmission

> All CCK+BC to PC pairs (n = 6) facilitate at 1 to 40 Hz; at 80 Hz and above and under EPI the
> facilitation of the 1st to 5th APs turns into depression later in the train.

> Between CCK+ cells IPSC amplitudes facilitate at 10 and 40 Hz, while at 160 Hz they show no consistent
> change or a mixture of depression and facilitation.

> Of eight CCK+DTI to PC pairs, four respond at low frequencies (1 to 40 Hz) or to RIP; in these the first
> AP of trains evokes an IPSC only from about the fourth sweep, transmission facilitates at 10 Hz but
> depresses at 40 and 160 Hz; the other four transmit, with depression, only under EPI.

In [ ]:
# fig6-cckbc-pc-trains, fig10-cck-cck-frequency, cckdti-pc-transmission: CCK+ trains.
for k in [("a", 0), ("a", 3), ("b", 0), ("b", 3)]:
    t = TR[k]
    d = t[t.type == "CCK+BC to PC"]
    low = d[d.freq.isin([1, 5, 10, 20, 40])].pivot_table(index="pair", columns="freq", values="mean_2_10")
    fac_pairs_with1 = ((low > 1) | low.isna()).all(axis=1)   # judged at the frequencies each pair was recorded at
    hp = {}
    for proto in ["80 Hz", "160 Hz", "320 Hz", "EPI"]:
        dd = d[d.protocol == proto]
        if len(dd) == 0:
            continue
        m = np.nanmedian(np.vstack([x[:max(len(x) for x in dd.rel)] for x in dd.rel]), 0)
        last = m[9] if proto != "EPI" else m[-1]   # the 10th AP of a train, the 34th of EPI (NaN under M4 a: "AR")
        hp[proto] = dict(n=len(dd), peak_2_5=round(float(np.nanmax(m[1:5])), 3), last=round(float(last), 3),
                         holds=bool(np.nanmax(m[1:5]) > 1 and last < 1))
    print(f"\nM4 {k[0]}, first {k[1]} sweeps left out. CCK+BC to PC, mean relative amplitude of APs 2 to 10 per pair:")
    print(low.round(3).to_string())
    print(f"  pairs facilitating at every frequency 1 to 40 Hz they were recorded at: {int(fac_pairs_with1.sum())} of {len(low)}")
    print("  80 Hz and above, median over pairs:", hp)
    # fig10-cck-cck-frequency
    c = t[t.type.isin(["CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI"])]
    mm = c.groupby("freq").mean_2_10.median()
    c160 = c[c.freq == 160].mean_2_10
    print(f"  CCK+ to CCK+ pairs: median mean APs 2 to 10 {mm.round(3).to_dict()} (n {c.groupby('freq').size().to_dict()}); "
          f"at 160 Hz pairs above 1: {int((c160 > 1).sum())}, below 1: {int((c160 < 1).sum())}")
    # cckdti-pc-transmission: responding pairs and their train pattern
    dt = t[t.type == "CCK+DTI to PC"]
    print("  CCK+DTI to PC, mean APs 2 to 10 by pair and protocol; last-AP relative amplitude under EPI:")
    print(dt.pivot_table(index="pair", columns="protocol", values="mean_2_10").round(3).to_string())
    print("  EPI 34th (last) AP:", dt[dt.protocol == "EPI"].set_index("pair").last.round(3).to_dict(),
          "| last AP with an amplitude:", dt[dt.protocol == "EPI"].set_index("pair").last_finite.round(3).to_dict())

# cckdti-pc-transmission: which pairs transmit at 1 to 40 Hz or to RIP, and the first sweep with a uIPSC to
# the first AP of each train (M2 a), mean over the train recordings of the responding pairs.
rows = []
for key in sorted(k for k in PAIRS if k[0] == "CCK+DTI to PC"):
    pr = PAIRS[key]
    low = [p for p in ["RIP", "1 Hz", "5 Hz", "10 Hz", "20 Hz", "40 Hz"] if p in pr and pr[p] in EPA]
    tp_low = {p: round(float(np.nanmean(EPA[pr[p]].success[:, 1 if p == "RIP" else 0:])), 3) for p in low}
    first = {}
    for p in ["1 Hz", "5 Hz", "10 Hz", "20 Hz", "40 Hz", "80 Hz", "160 Hz", "320 Hz"]:
        if p in pr and pr[p] in EPA:
            s = EPA[pr[p]].success[:, 0]
            first[p] = int(np.flatnonzero(s > 0)[0] + 1) if (s > 0).any() else None
    rows.append(dict(pair=key[1], desc_pair=fmap.get(key, ["?"])[0], low_protocols=low, tp_low=tp_low,
                     transmits_low=any(v > 0 for v in tp_low.values()), first_success_sweep=first))
DTI = pd.DataFrame(rows)
print(DTI.to_string())
resp = DTI[DTI.transmits_low]
fs = [v for d in resp.first_success_sweep for v in d.values() if v]
print(f"pairs transmitting at 1 to 40 Hz or RIP: {len(resp)} of {len(DTI)}; first sweep with a uIPSC to the first AP, "
      f"mean over {len(fs)} train recordings of these pairs: {np.mean(fs):.2f} (values {fs})")
fs40 = [v for d in resp.first_success_sweep for p, v in d.items() if v and int(p.split()[0]) <= 40]
print(f"variant, trains at 1 to 40 Hz only (the paper's low frequencies): mean {np.mean(fs40):.2f} over {len(fs40)} recordings (values {fs40})")

**Result: fig6-cckbc-pc-trains not reproduced.** Of the 6 CCK+BC to PC pairs with trains only 1 has a
mean relative amplitude of APs 2 to 10 above 1 at every frequency from 1 to 40 Hz it was recorded at
(M4 a and b; 0 of 6 without the first three sweeps); two pairs depress at every frequency (Pair01, 0.51
to 0.84; Pair02, 0.61 to 0.89). The last AP is the 10th of a train and the 34th of EPI. The high-frequency
pattern holds at all four protocols under M4 b (peak at APs 2 to 5 1.03 to 1.10; last AP 0.69 at 80 Hz,
0.26 at 160 Hz, -0.12 at 320 Hz, 0.15 under EPI), and under M4 a at 160 and 320 Hz (last AP 0.47, 0.16)
but not at 80 Hz (1.13); under M4 a the 34th EPI AP has no amplitude in the tables (marked `AR`). The
outcome fails on the facilitation part.

**Result: fig10-cck-cck-frequency reproduced.** Under M4 a the median over CCK+ to CCK+ pairs of the mean
relative amplitude of APs 2 to 10 is 1.559 at 10 Hz (6 pairs) and 2.153 at 40 Hz (9 pairs); at 160 Hz 5
pairs are above 1 and 4 below. M4 b and the variants agree on the 10 and 40 Hz facilitation and the
mixture at 160 Hz.

**Result: cckdti-pc-transmission not reproduced.** The data set holds 7 CCK+DTI to PC pairs, not 8. Five
transmit at 1 to 40 Hz or to RIP (expected 4, tolerance 0), and the first sweep with a uIPSC to the first
AP of a train is on average 2.0 (values 1 to 4 over 11 train recordings, 160 Hz included; expected 4 plus
or minus 0.4); over the 7 trains at 40 Hz or below it is 2.57, also outside. The frequency pattern of the
responding pairs holds (median mean of APs 2 to 10 above 1 at 10 Hz, below 1 at 40 and 160 Hz, M4 a). Of
the two pairs that transmit only under EPI, Pair07 depresses at the 34th AP (0.173, M4 b) and Pair06 does
not (1.012, M4 b; under M4 a the last AP with an amplitude is 0.584 and 3.675).

## Asynchronous release

The reconstruction of the Methods (template from the 11th IPSCs, synchronous trace, 100 Hz low-pass,
difference). Onset readings of the claims file: (a) immediate asynchronous charge above 2 SD of the
baseline charge for three APs in a row; (b) the current no longer returns to within 10% of the first
IPSC peak before the next AP.

In [ ]:
# Asynchronous release (paper, Methods, Asynchronous charge calculation; claims fig6-cckbc-async-onset,
# cckdti-async-onset, async-charge-epi-vs-160, decay-epi-vs-160, fig10-*). Per pair: the template, Eq. 6
# fitted to the average of all 11th IPSCs of its 10-AP trains (where the pair has no train, to the average
# single AP before the EPI burst: a deviation). Per recording: the sweep-averaged trace, the synchronous
# reconstruction (templates at each AP plus the delay, scaled to the averaged amplitude at that AP measured
# from the current just before it, M4 b), both low-pass filtered at 100 Hz (fourth-order zero-phase
# Butterworth, the claims file's named standard choice), their difference the asynchronous current.
def pair_template(key):
    snips = []
    for proto in TRAIN_PROTOS[1:]:
        p = PAIRS[key].get(proto)
        if p is None or p not in EPA:
            continue
        tr = trials(p, [10])
        if tr:
            tt, ys = hc7.snippets(RECS[p], [s for s, *_ in tr], [t for _, t, *_ in tr], pre=10, post=80)
            snips.append(ys)
    src = "11th AP of the trains"
    if not snips and "EPI" in PAIRS[key] and PAIRS[key]["EPI"] in EPA:
        p = PAIRS[key]["EPI"]
        tr = trials(p, [0])
        tt, ys = hc7.snippets(RECS[p], [s for s, *_ in tr], [t for _, t, *_ in tr], pre=10, post=80)
        snips.append(ys); src = "single AP before EPI"
    if not snips:
        return None, None
    y = np.nanmean(np.vstack(snips), 0)
    return hc7.fit_template(tt, y), src

def analyse(key, proto, template, scale="b"):
    p = PAIRS[key].get(proto)
    if p is None or p not in EPA or template is None:
        return None
    rec, e = RECS[p], EPA[p]
    times, present = hc7.stim_template(rec, proto)
    sweeps = [sw - 1 for sw in e.sweeps if sw - 1 < len(rec.post) and present[sw - 1].all()]
    if not sweeps:
        return None
    fs = rec.rate
    first = 1 if proto == "EPI" else 0
    t0 = times[first]
    base = (times[0] - 250, times[0] - 5)
    y = hc7.mean_trace(rec, sweeps, base)
    a = hc7.local_amplitudes(y, fs, times)
    scale_amps = a.copy()
    if scale == "a":   # the authors' averaged amplitudes (M4 a); where the table has none ("AR"), reading b
        ea = amps(key, proto, "a")
        scale_amps = np.where(np.isfinite(ea), ea, a)
    recon = hc7.reconstruct(len(y), fs, times, scale_amps, template)
    resid = hc7.butter_lowpass(y, fs) - hc7.butter_lowpass(recon, fs)
    # the stimulation train: the 34 APs of the EPI burst, or the first 10 APs of a 10-AP train (its 11th AP,
    # 1 s later, is not part of the train)
    train = times[first:] if proto == "EPI" else times[:10]
    isi = np.diff(train).mean() if len(train) > 1 else 100.0
    # immediate charge at each train AP: the interval to the next AP (the mean ISI after the last)
    edges = list(train) + [train[-1] + np.diff(train)[-1]]
    imm = np.array([hc7.window_charge(resid, fs, edges[j], edges[j + 1]) for j in range(len(train))])
    # noise: the same-length windows of the residual over the pre-train baseline
    if proto == "EPI":
        nb0, nb1 = times[0] + 300, t0 - 5
    else:
        nb0, nb1 = times[0] - 250, times[0] - 5
    w = np.diff(train)
    wins = []
    for wl in np.unique(np.round(w, 1)):
        k = np.arange(nb0, nb1 - wl, wl)
        wins += [hc7.window_charge(resid, fs, s, s + wl) for s in k]
    sd = np.std(wins) if len(wins) > 2 else np.nan
    above = imm > 2 * sd
    onset_a = next((j + 1 for j in range(len(train) - 2) if above[j:j + 3].all()), None)
    # reading b: from this AP on, the current before the next AP stays above 10% of the first IPSC's peak
    r = fs / 1e3
    pre_next = np.array([y[int(round((tn - 1.3) * r)):int(round((tn - 0.8) * r))].mean() for tn in edges[1:]])
    not_ret = pre_next > 0.1 * a[first]
    onset_b = next((j + 1 for j in range(len(train)) if not_ret[j:].all()), None)
    t_last = train[-1]   # the last AP of the EPI burst, or the 10th AP of a train
    delayed_a = hc7.window_charge(resid, fs, t_last, t_last + 500)
    delayed_b = hc7.window_charge(y, fs, t_last, t_last + 500)
    total = hc7.window_charge(y, fs, train[0], train[9] + 500) if len(train) >= 10 else np.nan
    # duration of delayed release: residual back within 2 SD of its baseline (sample SD over the baseline)
    sd_s = resid[int(nb0 * r):int(nb1 * r)].std()
    after = resid[int(round(t_last * r)):int(round((t_last + 1000) * r))]
    back = np.flatnonzero(np.abs(after) < 2 * sd_s)
    dur = back[0] / r if len(back) else np.nan
    return dict(type=key[0], pair=key[1], protocol=proto, scale=scale, n_sweeps=len(sweeps), onset_a=onset_a, onset_b=onset_b,
                onset_AR=(e.ar_first - first + 1) if e.ar_first is not None else None,
                imm=imm, sd=sd, delayed_a=delayed_a, delayed_b=delayed_b, total=total, duration=dur,
                tau_a=hc7.decay_after(y, fs, t_last, "a"), tau_b=hc7.decay_after(y, fs, t_last, "b"),
                first_amp=a[first])
def at_bound(tpl):
    """A template fit at a bound of hc7.fit_template (A 1e5, tau_r 0.05 or 20 ms, tau_d 0.3 or 200 ms)."""
    d, A, tr, td = tpl
    return bool(A >= 0.99e5 or tr <= 0.051 or tr >= 19.9 or td <= 0.301 or td >= 199)
TEMPL, AS = {}, []
for key in sorted(PAIRS):
    tpl, src = pair_template(key)
    TEMPL[key] = (tpl, src)
    for proto in ["EPI"] + TRAIN_PROTOS[1:]:
        for scale in "ab":
            res = analyse(key, proto, tpl, scale)
            if res:
                # a template at a bound gives a meaningless reconstruction: the measures that use it
                # (rule a, immediate and delayed asynchronous charge, duration) are dropped for the pair
                res["template_at_bound"] = at_bound(tpl)
                if res["template_at_bound"]:
                    res.update(onset_a=None, imm=np.full(len(res["imm"]), np.nan), sd=np.nan, delayed_a=np.nan, duration=np.nan)
                AS.append(res)
AS = pd.DataFrame(AS)
print("templates (delay ms, A, tau_r, tau_d) and their source, by pair:")
for k, (tpl, src) in TEMPL.items():
    print(" ", k, src, None if tpl is None else np.round(tpl, 2), "AT A BOUND, reconstruction not used" if tpl is not None and at_bound(tpl) else "")
print("pairs whose template is at a bound (reconstruction-based measures dropped):",
      [k for k, (tpl, _) in TEMPL.items() if tpl is not None and at_bound(tpl)])
print("\nrecordings analysed:", AS[AS.scale == "b"].groupby(["type", "protocol"]).size().unstack().to_string())

In [ ]:
# fig6-cckbc-async-onset and cckdti-async-onset: onset AP (train AP number) under EPI; the authors' "AR"
# marks in the _EPA tables are shown as a third column (an undocumented code, not a reading of the claims file).
# The reconstruction is scaled to the averaged amplitudes measured from the current just before each AP
# (M4 b, scale "b"); a variant scales it to the _EPA amplitudes (M4 a, scale "a"), which the tables lack
# after an "AR" mark, where M4 b is used.
show = ["pair", "onset_a", "onset_b", "onset_AR", "sd", "n_sweeps"]
print("onset by the charge rule (a), EPI, both scalings: median (range), pairs with an onset")
for g in IN_PC:
    for sc in "ab":
        v = AS[(AS.type == g) & (AS.protocol == "EPI") & (AS.scale == sc)].onset_a
        print(f"  {g}, scale {sc}: {v.median():.1f} ({v.min():.0f} to {v.max():.0f}), {v.notna().sum()} of {len(v)}")
AS_all, AS = AS, AS[AS.scale == "b"]
for g in ["CCK+BC to PC", "CCK+DTI to PC", "PV+BC to PC", "AAC to PC"]:
    d = AS[(AS.type == g) & (AS.protocol == "EPI")]
    print(f"\n{g}, EPI ({len(d)} pairs):")
    print(d[show].to_string())
    for c in ("onset_a", "onset_b", "onset_AR"):
        v = pd.to_numeric(d[c], errors="coerce").dropna()
        if len(v):
            print(f"  {c}: {len(v)} of {len(d)} pairs with an onset; median {v.median():.1f}, quartiles "
                  f"{v.quantile(.25):.1f} to {v.quantile(.75):.1f}, range {v.min():.0f} to {v.max():.0f}")
        else:
            print(f"  {c}: no pair with an onset")
print("\npairs with an onset under the 10-AP trains, by rule and type (scale b):")
tr10 = AS[AS.protocol != "EPI"]
print(tr10.assign(a=tr10.onset_a.notna(), b=tr10.onset_b.notna(), AR=tr10.onset_AR.notna())
      .groupby(["type", "protocol"])[["a", "b", "AR"]].sum().astype(int).to_string())
print("\nCCK+DTI to PC, trains at 160 and 320 Hz:")
print(tr10[(tr10.type == "CCK+DTI to PC") & tr10.protocol.isin(["160 Hz", "320 Hz"])][show + ["protocol"]].to_string())

# the CCK+ to CCK+ pairs (fig10-*): onset in the 10-AP trains by rule
cc = AS[AS.type.isin(["CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI"])]
print("\nCCK+ to CCK+ pairs, onset by rule a and b per train (scale b):")
print(cc.pivot_table(index=["type", "pair"], columns="protocol", values=["onset_a", "onset_b"], aggfunc="first").to_string())
AS = AS_all

# The criteria, under each onset rule and each scaling of the reconstruction.
def onsets(sc, g, protos, rule):
    d = AS[(AS.scale == sc) & (AS.type == g) & AS.protocol.isin(protos)]
    return pd.to_numeric(d[f"onset_{rule}"], errors="coerce"), d
print("\nfig6-cckbc-async-onset: CCK+BC to PC, EPI median onset 10 to 12, all onsets 9 to 21, no CCK+BC to PC pair with an onset in a 10-AP train")
for sc in "ab":
    for rule in "ab":
        v, d = onsets(sc, "CCK+BC to PC", ["EPI"], rule)
        w, dt_ = onsets(sc, "CCK+BC to PC", TRAIN_PROTOS[1:], rule)
        tr_pairs = sorted(set(dt_.pair[w.notna()]))
        vv = v.dropna()
        ok = bool(len(vv) and 10 <= vv.median() <= 12 and vv.between(9, 21).all() and not tr_pairs)
        print(f"  scale {sc}, rule {rule}: EPI onsets in {len(vv)} of {len(v)} pairs, median {vv.median() if len(vv) else np.nan}, "
              f"range {vv.min() if len(vv) else np.nan} to {vv.max() if len(vv) else np.nan}; pairs with an onset in some 10-AP train: "
              f"{len(tr_pairs)} of {dt_.pair.nunique()}; holds: {ok}")
print("\ncckdti-async-onset: CCK+DTI to PC, EPI median within 6.3 to 7.7, trains (a: 160 and 320 Hz pooled; b: 320 Hz) median within 6 to 8, below the CCK+BC EPI median")
for sc in "ab":
    for rule in "ab":
        v = onsets(sc, "CCK+DTI to PC", ["EPI"], rule)[0].dropna()
        vb = onsets(sc, "CCK+BC to PC", ["EPI"], rule)[0].dropna()
        for tr_reading, protos in (("a", ["160 Hz", "320 Hz"]), ("b", ["320 Hz"])):
            w, dt_ = onsets(sc, "CCK+DTI to PC", protos, rule)
            ww = w.dropna()
            ok = bool(len(v) and len(ww) and 6.3 <= v.median() <= 7.7 and 6 <= ww.median() <= 8 and v.median() < vb.median())
            print(f"  scale {sc}, rule {rule}, trains {tr_reading}: EPI median {v.median() if len(v) else np.nan} ({len(v)} pairs); "
                  f"trains median {ww.median() if len(ww) else np.nan} ({len(ww)} of {len(w)} recordings with an onset, values "
                  f"{ww.astype(int).tolist()}); CCK+BC EPI median {vb.median() if len(vb) else np.nan}; holds: {ok}")

Under EPI, rule (a) gives an onset at the 2nd to 4th AP in CCK+BC to PC pairs (median 2.0, 10 of 10
pairs), and the same in 6 of 7 PV+BC to PC and 8 of 8 AAC to PC pairs, which the paper describes as
synchronous. Rule (b) gives the 1st AP for every PV+BC, AAC and CCK+BC to PC pair. Scaling the
reconstruction to the authors' amplitudes changes little (CCK+BC median 3.5, PV+BC 2.5). In the 10-AP
trains, now cut at the 10th AP, both rules still flag the 160 Hz train of every CCK+BC to PC pair and of
most PV+ pairs (rule a: 5 of 6 PV+BC to PC, 8 of 8 AAC to PC, 12 of 12 PV+BC to PV+BC). Two CCK+DTI to PC
templates are at a bound of the fit (Pair04, Pair06), so rule (a) is not available for them. The
authors' own `AR` marks in the `_EPA` tables, which are absent from every PV+BC and AAC table, give for
CCK+BC to PC under EPI a median onset of 12 (quartiles 10 to 12.8, range 8 to 19; paper 11, 10 to 14,
range 10 to 19), for CCK+DTI to PC a median of 8 (7 to 9, range 5 to 10; paper 7, 7 to 9, range 7 to 13),
and at 160 Hz for CCK+DTI to PC 6 to 8 (paper 7, 6 to 8). The code is undocumented and is not a rule of
the claims file, so these are shown and do not decide.

**Result: fig6-cckbc-async-onset not reproduced.** Under both onset rules of the claims file and both
scalings, the median CCK+BC to PC EPI onset is 1.0 to 3.5 (expected 10 to 12), and all 6 CCK+BC to PC
pairs with trains have an onset in some 10-AP train. The rules also flag the PV+ pairs the paper calls
synchronous, so they do not appear to be the authors' procedure; whether the claims file's rules can
stand for it is asked of a reader, and how the onset was found is a question for the paper's authors.

**Result: cckdti-async-onset not reproduced.** The CCK+DTI to PC EPI median onset is 3.0 under rule (a)
(5 pairs; 4.5 with the authors' amplitudes) and 1.0 under rule (b), against 7 (6.3 to 7.7); at 160 Hz
rule (a) gives onsets at the 2nd, 7th and 2nd AP (median 2.0) and rule (b) at the 1st in all 4 pairs
(expected 6 to 8). No CCK+DTI to PC pair has a 320 Hz train, so the reading "320 Hz alone" cannot be
computed. The CCK+DTI median is not below the CCK+BC median under any rule (3.0 against 2.0, 4.5 against
3.5, 1.0 against 1.0).

## async-charge-epi-vs-160 and decay-epi-vs-160

> At CCK+BC to PC pairs the EPI protocol triggers a larger asynchronous charge after the last AP than 160
> Hz trains, while the charge is unchanged between the two patterns at PV+BC, AAC and CCK+DTI to PC pairs.

> The decay time constant after the last AP lengthens under EPI compared with 160 Hz trains, from 13.01 to
> 103.65 ms at CCK+BC to PC pairs and from 36.82 to 138.71 ms at CCK+DTI to PC pairs.

In [ ]:
# async-charge-epi-vs-160 and decay-epi-vs-160: the 500 ms after the last AP of the EPI burst and of the
# 160 Hz train. Charge reading a: the delayed asynchronous charge (100 Hz low-pass original minus
# reconstruction); reading b: the area under the mean trace in that window, measured from the
# pre-stimulation baseline (the sweeps end too soon for an equal window at least 500 ms later, a deviation).
# For the 160 Hz train the window and the decay fit start at the 10th AP; the 11th AP comes 1 s later, after
# the 500 ms window.
ASb = AS[AS.scale == "b"]
rows = []
for g in IN_PC:
    for c in ("delayed_a", "delayed_b", "tau_a", "tau_b"):
        x = ASb[(ASb.type == g) & (ASb.protocol == "EPI")][c].dropna()
        y = ASb[(ASb.type == g) & (ASb.protocol == "160 Hz")][c].dropna()
        p = stats.mannwhitneyu(x, y, alternative="two-sided").pvalue if len(x) and len(y) else np.nan
        rows.append(dict(type=g, measure=c, n_EPI=len(x), n_160=len(y), median_EPI=x.median(), median_160=y.median(), p=p))
CH = pd.DataFrame(rows)
print("charge in pC (delayed_*), decay time constant in ms (tau_*: a, fit to 500 ms; b, fit to 5% of the peak):")
print(CH.to_string(float_format=lambda x: f"{x:.4g}"))
for c in ("delayed_a", "delayed_b"):
    q = CH[CH.measure == c].set_index("type")
    cck = q.loc["CCK+BC to PC"]
    ok = cck.p < 0.05 and cck.median_EPI > cck.median_160
    absent = {g: bool(q.loc[g, "p"] >= 0.05) for g in ["PV+BC to PC", "AAC to PC", "CCK+DTI to PC"]}
    print(f"{c}: CCK+BC EPI larger at p < 0.05: {ok}; no difference (p >= 0.05) at {absent}")
for c in ("tau_a", "tau_b"):
    q = CH[CH.measure == c].set_index("type")
    meds = [q.loc["CCK+BC to PC", "median_160"], q.loc["CCK+BC to PC", "median_EPI"], q.loc["CCK+DTI to PC", "median_160"], q.loc["CCK+DTI to PC", "median_EPI"]]
    nw = sum(within(v, e) for v, e in zip(meds, [13.01, 103.65, 36.82, 138.71]))
    print(f"{c}: medians CCK+BC 160 Hz {meds[0]:.2f}, EPI {meds[1]:.2f}; CCK+DTI 160 Hz {meds[2]:.2f}, EPI {meds[3]:.2f} "
          f"({nw} of 4 within 10%); p CCK+BC {q.loc['CCK+BC to PC', 'p']:.3g}, CCK+DTI {q.loc['CCK+DTI to PC', 'p']:.3g}")
print("\nper pair, all IN to PC types (charge in pC, decay in ms; the 160 Hz values after the 10th AP):")
ASa = AS[AS.scale == "a"]
print("variant, reconstruction scaled to the _EPA amplitudes: delayed charge (reading a), Mann-Whitney p EPI against 160 Hz:",
      {g: round(stats.mannwhitneyu(ASa[(ASa.type == g) & (ASa.protocol == "EPI")].delayed_a,
                                   ASa[(ASa.type == g) & (ASa.protocol == "160 Hz")].delayed_a).pvalue, 4) for g in IN_PC})
print(ASb[ASb.type.isin(IN_PC) & ASb.protocol.isin(["EPI", "160 Hz"])]
      .sort_values(["type", "pair", "protocol"])[["type", "pair", "protocol", "delayed_a", "delayed_b", "tau_a", "tau_b"]].round(3).to_string())

**Result: async-charge-epi-vs-160 not reproduced.** With the 500 ms window after the 10th AP of the 160
Hz train (the earlier run used the isolated 11th AP), CCK+BC to PC pairs carry more charge after EPI than
after the train under both readings, but not significantly (reading a, delayed asynchronous charge, 4.44
against 1.18 pC; reading b, area under the mean trace, 4.67 against 1.61 pC; p 0.0934 for both, 10 EPI
and 6 train recordings). The CCK+BC part fails, so the claim is not reproduced under either reading. The
absence parts: under reading b PV+BC (p 0.138), AAC (p 0.130) and CCK+DTI (p 0.649) show no difference;
under reading a PV+BC (p 0.0513) and CCK+DTI (p 1; 5 EPI and 3 train recordings, two pairs without a
usable template) show none and AAC does (p 0.0148).

**Result: decay-epi-vs-160 partially reproduced.** Fitted after the 10th AP of the 160 Hz train, the decay
after the last AP is longer under EPI at both CCK+BC (p 0.00025) and CCK+DTI (p 0.0121) pairs with the fit
to 500 ms (reading a), with medians 15.07 against 13.01 ms and 88.27 against 103.65 ms (CCK+BC) and 38.51
against 36.82 ms and 104.13 against 138.71 ms (CCK+DTI): 1 of 4 within tolerance. With the fit to 5% of
the peak (reading b) the EPI medians are within tolerance (96.05 against 103.65; 125.65 against 138.71 ms)
and the 160 Hz medians are not (14.82 against 13.01; 44.37 against 36.82 ms), and the CCK+DTI test misses
(p 0.0727). Both tests hold with EPI longer under reading a while some medians fall outside tolerance,
the criterion's partial.

## fig10-cck-cck-heterogeneity, fig10-cck-cck-async, fig10bd-be-async-charge

> Transmission between CCK+ cells is heterogeneous; all connections facilitate, four of nine pairs release
> only synchronously while five show frequency-dependent strong asynchronous release, and both BC and DTI
> pairs show either pattern.

> In the five CCK+ pairs with asynchronous release, release is synchronous at 10 Hz but asynchronous after
> the 5th AP at 40 and 160 Hz, and persists after the train for 69.8 to 339.6 ms, a duration independent of
> frequency; the total charge depends on the presynaptic frequency.

> Between CCK+ cells immediate asynchronous charge per AP rises from 10 to 40 Hz, starting near the 5th AP,
> and falls at higher frequencies; delayed asynchronous charge grows with frequency and saturates at 160 Hz.

The CCK+ to CCK+ pairs were recorded with 10, 40 and 160 Hz trains only, so the two readings of which
trains count (40 Hz and above, or 20 Hz and above) select the same trains, and the 320 Hz parts of
fig10bd-be-async-charge cannot be computed.

In [ ]:
# fig10-cck-cck-heterogeneity, fig10-cck-cck-async, fig10bd-be-async-charge: CCK+ to CCK+ pairs, 10-AP trains,
# onset rules (a) and (b) of fig6-cckbc-async-onset on the reconstruction scaled by M4 b (scale "b").
CC = ["CCK+BC to CCK+BC", "CCK+DTI to CCK+DTI"]
cc = AS[(AS.scale == "b") & AS.type.isin(CC) & (AS.protocol != "EPI")].copy()
cc["freq"] = cc.protocol.map(freq)
print("trains per CCK+ to CCK+ pair:", {f"{t} {p}": sorted(v) for (t, p), v in cc.groupby(["type", "pair"]).freq.apply(list).items()})
print("CCK+ to CCK+ pairs whose template is at a bound (rule a and charges not available):",
      sorted(set(cc[cc.template_at_bound].pair + " " + cc[cc.template_at_bound].type)))
# facilitation part: per pair the largest relative amplitude at APs 2 to 10 of the 10 or 40 Hz train
fac = {}
for rd in "ab":
    t = TR[(rd, 0)]
    d = t[t.type.isin(CC) & t.freq.isin([10, 40])]
    fac[rd] = d.groupby(["type", "pair"]).max_2_10.max()
    print(f"M4 {rd}: largest relative amplitude of APs 2 to 10 at 10 or 40 Hz per pair: {fac[rd].round(3).to_dict()}; "
          f"all above 1: {bool((fac[rd] > 1).all())}")
fac_ok = bool((fac["a"] > 1).all() or (fac["b"] > 1).all())

# fig10-cck-cck-heterogeneity: a pair has asynchronous release when the rule gives an onset in some train at or
# above 40 Hz (reading a) or 20 Hz (reading b)
HET = {}
print("\nfig10-cck-cck-heterogeneity (expected 45 to 66% of pairs with asynchronous release, both types in each group):")
for rule in "ab":
    for rd, fmin in (("a", 40), ("b", 20)):
        d = cc[cc.freq >= fmin]
        has = d.groupby(["type", "pair"])[f"onset_{rule}"].apply(lambda s: bool(s.notna().any()))
        HET[(rule, rd)] = has
        frac = has.mean()
        t_async = {t for (t, p), v in has.items() if v}
        t_sync = {t for (t, p), v in has.items() if not v}
        ok = bool(0.45 <= frac <= 0.66 and t_async == set(CC) and t_sync == set(CC) and fac_ok)
        print(f"  rule {rule}, trains {rd}: {int(has.sum())} of {len(has)} pairs ({frac:.0%}) with asynchronous release; "
              f"types with it {sorted(t_async)}, without {sorted(t_sync)}; holds: {ok}")
        print("    per pair:", {f"{t[:6]} {p}": v for (t, p), v in has.items()})

# fig10-cck-cck-async and fig10bd-be-async-charge, for the pairs with asynchronous release under each rule
cc["imm_5_10"] = [np.nansum(x[4:10]) if np.isfinite(x[4:10]).any() else np.nan for x in cc.imm]
print("\nper pair and train: onsets, immediate charge summed over APs 5 to 10 and delayed charge (pC, reading a "
      "of the claims file: residual after the 10th AP), total charge (pC), duration of delayed release (ms):")
print(cc.sort_values(["type", "pair", "freq"])[["type", "pair", "freq", "onset_a", "onset_b", "imm_5_10", "delayed_a",
                                               "total", "duration"]].round(3).to_string())
print("\nmedian immediate charge per AP (pC) over all CCK+ to CCK+ pairs, APs 1 to 10:")
for f, d in cc.groupby("freq"):
    print(f"  {f} Hz ({len(d)} pairs):", np.round(np.nanmedian(np.vstack(d.imm.values), 0), 4))
for rule in "ab":
    has = HET[(rule, "a")]
    keys = [k for k, v in has.items() if v]
    if not keys:
        print(f"\nrule {rule}: no pair with asynchronous release; fig10-cck-cck-async and fig10bd-be-async-charge have no pairs")
        continue
    d = cc[[(t, p) in keys for t, p in zip(cc.type, cc.pair)]]
    # fig10-cck-cck-async: onset after the 5th AP at 40 and 160 Hz and none at 10 Hz, per pair
    rows = []
    for (t, p), g in d.groupby(["type", "pair"]):
        on = g.set_index("freq")[f"onset_{rule}"]
        late = all(pd.notna(on.get(f)) and on.get(f) > 5 for f in (40, 160) if f in on.index)
        none10 = pd.isna(on.get(10)) if 10 in on.index else True
        rows.append(dict(pair=f"{t[:6]} {p}", onsets=on.to_dict(), after_5th_at_40_160=late, none_at_10=none10))
    A10 = pd.DataFrame(rows)
    n_ok = int((A10.after_5th_at_40_160 & A10.none_at_10).sum())
    dur = d.duration.dropna()
    dur160 = d[d.freq == 160].duration.dropna()
    in_range = bool(len(dur) and dur.between(62.8, 373.6).all())
    gt = [g.total.dropna().values for _, g in d.groupby("freq")]
    gd = [g.duration.dropna().values for _, g in d.groupby("freq")]
    kw_t = stats.kruskal(*gt).pvalue if sum(len(x) > 0 for x in gt) > 1 else np.nan
    kw_d = stats.kruskal(*[x for x in gd if len(x)]).pvalue if sum(len(x) > 0 for x in gd) > 1 else np.nan
    print(f"\nrule {rule}: {len(keys)} pairs with asynchronous release")
    print(A10.to_string())
    print(f"  fig10-cck-cck-async: onset after the 5th AP at 40 and 160 Hz and none at 10 Hz in {n_ok} of {len(A10)} pairs "
          f"(at least 80% needed); durations {np.round(dur.min(), 1) if len(dur) else np.nan} to {np.round(dur.max(), 1) if len(dur) else np.nan} ms "
          f"({len(dur)} values; all within 62.8 to 373.6: {in_range}); Kruskal-Wallis across 10, 40, 160 Hz, total charge p {kw_t:.3g} "
          f"(medians {d.groupby('freq').total.median().round(3).to_dict()}), duration p {kw_d:.3g} "
          f"(medians {d.groupby('freq').duration.median().round(1).to_dict()}); at 160 Hz alone the durations are "
          f"{np.round(dur160.min(), 1)} to {np.round(dur160.max(), 1)} ms, {int(dur160.between(62.8, 373.6).sum())} of {len(dur160)} within 62.8 to 373.6")
    mi = d.groupby("freq").imm_5_10.median()
    md = d.groupby("freq").delayed_a.median()
    print(f"  fig10bd-be-async-charge: median immediate charge over APs 5 to 10 {mi.round(4).to_dict()} pC "
          f"(40 above 10 Hz: {bool(mi.get(40, np.nan) > mi.get(10, np.nan))}); median delayed charge {md.round(4).to_dict()} pC "
          f"(no step from 10 to 160 Hz lower than the previous by more than 10%: "
          f"{bool(md.get(40, np.nan) >= 0.9 * md.get(10, np.nan) and md.get(160, np.nan) >= 0.9 * md.get(40, np.nan))}); "
          f"no 320 Hz train, so the 320 Hz parts cannot be computed")

**Result: fig10-cck-cck-heterogeneity not reproduced.** Both onset rules mark asynchronous release in all
9 CCK+ to CCK+ pairs with a train at 40 Hz or above (100%, expected 45 to 66%), under both readings of
which trains count, which select the same trains here; no pair is purely synchronous. The facilitation part
holds: every pair has a relative amplitude above 1 at some AP of its 10 or 40 Hz train (largest 1.05 to
12.46 under M4 a, 1.39 to 5.71 under M4 b).

**Result: fig10-cck-cck-async not reproduced.** With all 9 pairs counted as asynchronous, no pair has its
onset after the 5th AP at both 40 and 160 Hz with none at 10 Hz (0 of 9 under both rules; at 160 Hz the
onset is at the 2nd to 5th AP under rule a and the 1st under rule b). The durations of delayed release at
160 Hz are 24.0 to 222.4 ms, 4 of 9 within 62.8 to 373.6 ms, and over all trains they depend on the
frequency (Kruskal-Wallis p 0.000884), as the total charge does (p 0.0444, medians 0.358, 0.749 and
2.891 pC at 10, 40 and 160 Hz).

**Result: fig10bd-be-async-charge blocked (data_missing).** The CCK+ to CCK+ pairs have no 320 Hz train, so
the parts that compare 320 Hz with 40 and 160 Hz cannot be computed. The other parts hold over the 9 pairs:
the median immediate charge over APs 5 to 10 rises from 0.037 pC at 10 Hz to 0.371 pC at 40 Hz (and 0.837
pC at 160 Hz), and the median delayed charge rises from 0.012 to 0.102 to 1.319 pC at 10, 40 and 160 Hz.

## Models: fig11-model-pv-udr, fig11-model-cck-facilitation, table4-model-parameters, fig11c-parameter-differences, fig11-model-prediction

The four models of Eqs. 1 to 5 fitted per connection type, shared tau_r (and tau_e, a_e, k_e) and
per-connection U0, Imax (and tau_f); readings of the error, (a) unweighted, (b) each connection's errors
divided by its first-IPSC amplitude. Four starting points per fit from a generator seeded in this
section (a standard choice). The fits take a few minutes.

In [ ]:
# fig11-*: the four short-term plasticity models fitted per connection type to the mean amplitudes (pA,
# M4 a) of the 10-AP trains at 1 to 320 Hz with the 11th AP, all frequencies together (Results, Simple
# mathematical models, paragraph 3; Table 4 legend). Test data: the RIP and EPI amplitudes (Fig. 11 legend).
MODEL_TYPES = ["PV+BC to PV+BC", "PV+BC to PC", "AAC to PC", "CCK+BC to PC"]
def model_data(g, protos, reading="a", ks=None):
    """Rows of mean amplitudes and intervals; with `ks`, the connection index is the position in ks (the
    training connections), and connections without these protocols are left out."""
    dts, ys, conn, firsts, keys = [], [], [], [], []
    if ks is None:
        ks = [k for k in sorted(PAIRS) if k[0] == g and any(p in PAIRS[k] and PAIRS[k][p] in EPA for p in protos)]
    for ci, key in enumerate(ks):
        for proto in protos:
            p = PAIRS[key].get(proto)
            if p is None or p not in EPA:
                continue
            a = amps(key, proto, reading)
            times, _ = hc7.stim_template(RECS[p], proto)
            dt = np.r_[np.nan, np.diff(times)] / 1e3
            row = np.full(36, np.nan); row[:len(a)] = a
            drow = np.full(36, np.nan); drow[:len(dt)] = dt
            ys.append(row); dts.append(drow); conn.append(ci)
        keys.append(key)
    y, dt, conn = np.array(ys), np.array(dts), np.array(conn, int)
    first = np.array([np.nanmax(y[conn == c][:, 0]) for c in conn])
    return dt, y, conn, keys, first
FITS = {}
t_fit = time.time()
rng = np.random.default_rng(11)
for g in MODEL_TYPES:
    dt, y, conn, keys, first = model_data(g, TRAIN_PROTOS)
    for wname, w in (("a", None), ("b", 1 / first)):
        for model in hc7.MODELS:
            FITS[(g, wname, model)] = hc7.fit_model(dt, y, conn, model, w, starts=4, rng=rng)
            FITS[(g, wname, model)]["keys"] = keys
print(f"{len(FITS)} fits in {time.time() - t_fit:.0f} s")
rows = []
for (g, wname, model), f in FITS.items():
    rows.append(dict(type=g, weighting=wname, model=model, n_conn=len(f["keys"]), n=f["n"], k=f["k"], sse=f["sse"], bic=f["bic"]))
FT = pd.DataFrame(rows)
print(FT.to_string(float_format=lambda x: f"{x:.4g}"))

In [ ]:
# fig11-model-pv-udr and fig11-model-cck-facilitation
for wname in "ab":
    print(f"\nweighting {wname}:")
    for g in MODEL_TYPES:
        f = {m: FITS[(g, wname, m)] for m in hc7.MODELS}
        dec = 100 * (1 - f["UDR dep"]["sse"] / f["TM dep"]["sse"])
        fac_gain = 100 * (1 - f["UDR dep+fac"]["sse"] / f["UDR dep"]["sse"])
        best = min(f, key=lambda m: f[m]["bic"])
        print(f"  {g}: SSE decrease with UDR {dec:.1f}%; facilitation added to UDR lowers SSE by {fac_gain:.1f}% "
              f"(BIC UDR dep {f['UDR dep']['bic']:.1f}, UDR dep+fac {f['UDR dep+fac']['bic']:.1f}); lowest BIC: {best}; "
              f"TM dep+fac SSE below TM dep: {f['TM dep+fac']['sse'] < f['TM dep']['sse']}")

# table4-model-parameters: the best model of Table 4 for each type (UDR dep for the PV+ types; TM dep+fac for CCK+BC)
EXP4 = {"PV+BC to PV+BC": dict(U0=0.31, Imax=410, tau_r=1.08), "PV+BC to PC": dict(U0=0.33, Imax=289, tau_r=1.62),
        "AAC to PC": dict(U0=0.26, Imax=646, tau_r=1.70), "CCK+BC to PC": dict(U0=0.15, tau_f=10.9, Imax=499, tau_r=0.071)}
TOL4 = {("PV+BC to PV+BC", "U0"): 0.03, ("PV+BC to PV+BC", "Imax"): 140, ("PV+BC to PV+BC", "tau_r"): 0.108,
        ("PV+BC to PC", "U0"): 0.09, ("PV+BC to PC", "Imax"): 36, ("PV+BC to PC", "tau_r"): 0.162,
        ("AAC to PC", "U0"): 0.04, ("AAC to PC", "Imax"): 182, ("AAC to PC", "tau_r"): 0.17,
        ("CCK+BC to PC", "U0"): 0.03, ("CCK+BC to PC", "tau_f"): 3.7, ("CCK+BC to PC", "Imax"): 209, ("CCK+BC to PC", "tau_r"): 0.0071}
def table4(F, wname):
    rows = []
    for g in MODEL_TYPES:
        f = F[(g, wname, "TM dep+fac" if g == "CCK+BC to PC" else "UDR dep")]
        got = dict(U0=np.median(f["U0"]), Imax=np.median(f["A"]), tau_r=f["tau_r"],
                   tau_f=np.median(f["tau_f"]) * 1e3 if f["tau_f"] is not None else np.nan)
        for q, e in EXP4[g].items():
            rows.append(dict(type=g, parameter=q, expected=e, tolerance=TOL4[(g, q)], fitted=got[q],
                             within=abs(got[q] - e) <= TOL4[(g, q)]))
    return pd.DataFrame(rows)
for wname in "ab":
    for g in MODEL_TYPES[:3]:
        f = FITS[(g, wname, "UDR dep")]
        print(f"  weighting {wname}, {g}: tau_e {f['tau_e'] * 1e3:.1f} ms, a_e {f['a_e']:.2f}, k_e {f['k_e']:.2f}/s")
    T4 = table4(FITS, wname)
    print(f"\nTable 4 parameters (tau_r in s, tau_f in ms), weighting {wname}: {int(T4.within.sum())} of 13 within tolerance")
    print(T4.to_string(float_format=lambda x: f"{x:.4g}"))
    fc = FITS[("CCK+BC to PC", wname, "UDR dep+fac")]
    print(f"  variant (b): CCK+BC tau_r in the comprehensive model {fc['tau_r']:.3g} s (Fig. 11 legend 12.9 s)")

# fig11c-parameter-differences: per-connection U0, tau_f and Imax of the comprehensive model (UDR dep+fac)
def fig11c(F, wname):
    P = []
    for g in MODEL_TYPES:
        f = F[(g, wname, "UDR dep+fac")]
        for i in range(len(f["keys"])):
            P.append(dict(type=g, U0=f["U0"][i], tau_f=f["tau_f"][i] * 1e3, Imax=f["A"][i]))
    P = pd.DataFrame(P)
    kw = {q: stats.kruskal(*[P[P.type == g][q] for g in MODEL_TYPES]).pvalue for q in ("U0", "tau_f", "Imax")}
    C = [("tau_f", "CCK+BC to PC", "AAC to PC", ">"), ("tau_f", "CCK+BC to PC", "PV+BC to PC", ">"),
         ("tau_f", "CCK+BC to PC", "PV+BC to PV+BC", ">"), ("U0", "CCK+BC to PC", "PV+BC to PV+BC", "<"),
         ("U0", "CCK+BC to PC", "PV+BC to PC", "<"), ("Imax", "PV+BC to PC", "others", "<")]
    rows = []
    for q, g1, g2, sgn in C:
        x = P[P.type == g1][q]
        y = P[P.type != g1][q] if g2 == "others" else P[P.type == g2][q]
        pv = stats.mannwhitneyu(x, y).pvalue
        ok = (x.median() > y.median()) if sgn == ">" else (x.median() < y.median())
        rows.append(dict(param=q, g1=g1, g2=g2, med1=x.median(), med2=y.median(), p=pv, p_bonf6=min(1, 6 * pv), holds=bool(ok and 6 * pv < 0.05)))
    imax_each = {g: stats.mannwhitneyu(P[P.type == "PV+BC to PC"].Imax, P[P.type == g].Imax).pvalue * 6
                 for g in MODEL_TYPES if g != "PV+BC to PC"}
    return P, kw, pd.DataFrame(rows), imax_each
for wname in "ab":
    P, kw, C11, imax_each = fig11c(FITS, wname)
    print(f"\nweighting {wname}: Kruskal-Wallis p {dict((k, round(v, 4)) for k, v in kw.items())}")
    print(C11.to_string(float_format=lambda x: f"{x:.4g}"))
    print("  Imax, PV+BC to PC against each other type, Bonferroni-corrected p:", {k: round(v, 4) for k, v in imax_each.items()})
    print("  medians:", P.groupby("type")[["U0", "tau_f", "Imax"]].median().round(3).to_dict())

# fig11-model-prediction: test on RIP and EPI with only A refitted per connection
for wname in "ab":
    out = {}
    for g in MODEL_TYPES:
        dt, y, conn, keys, first = model_data(g, ["RIP", "EPI"], ks=FITS[(g, wname, "TM dep")]["keys"])
        if len(y) == 0:
            continue
        out[g] = {m: hc7.refit_A(dt, y, conn, FITS[(g, wname, m)]) for m in hc7.MODELS}
    TE = pd.DataFrame(out).T
    print(f"\ntest mean squared error (pA^2), weighting {wname}:")
    print(TE.round(1).to_string())
    print("  lowest:", TE.idxmin(1).to_dict())

In [ ]:
# fig11 run-to-run variation (round-2 verifier): the 32 fits repeated with generators seeded 1, 2 and 3 (four
# starting points each), against the run above (seed 11). "best of 16" keeps, per type, weighting and model,
# the fit with the lowest error over all four runs (16 starting points).
def fig11_summary(F):
    rows = []
    for wname in "ab":
        for g in MODEL_TYPES:
            f = {m: F[(g, wname, m)] for m in hc7.MODELS}
            bic = {m: f[m]["bic"] for m in f}
            dt, y, conn, keys, first = model_data(g, ["RIP", "EPI"], ks=f["TM dep"]["keys"])
            te = {m: hc7.refit_A(dt, y, conn, f[m]) for m in f} if len(y) else {}
            rows.append(dict(weighting=wname, type=g, lowest_bic=min(bic, key=bic.get),
                             bic_tm_fac=bic["TM dep+fac"], bic_udr_fac=bic["UDR dep+fac"], bic_udr=bic["UDR dep"],
                             udr_dec=100 * (1 - f["UDR dep"]["sse"] / f["TM dep"]["sse"]),
                             fac_gain=100 * (1 - f["UDR dep+fac"]["sse"] / f["UDR dep"]["sse"]),
                             tm_fac_below_tm=f["TM dep+fac"]["sse"] < f["TM dep"]["sse"],
                             test_lowest=min(te, key=te.get) if te else "", te_tm=te.get("TM dep", np.nan)))
    return pd.DataFrame(rows)
RUNS = {11: FITS}
t_fit = time.time()
for seed in (1, 2, 3):
    rng_s = np.random.default_rng(seed)
    F = {}
    for g in MODEL_TYPES:
        dt, y, conn, keys, first = model_data(g, TRAIN_PROTOS)
        for wname, w in (("a", None), ("b", 1 / first)):
            for model in hc7.MODELS:
                F[(g, wname, model)] = hc7.fit_model(dt, y, conn, model, w, starts=4, rng=rng_s)
                F[(g, wname, model)]["keys"] = keys
    RUNS[seed] = F
print(f"{3 * len(FITS)} fits in {time.time() - t_fit:.0f} s")
BEST16 = {k: min((RUNS[s][k] for s in RUNS), key=lambda f: f["sse"]) for k in FITS}
SEEDS = pd.concat([fig11_summary(F).assign(run=f"seed {s}") for s, F in RUNS.items()]
                  + [fig11_summary(BEST16).assign(run="best of 16")])
print(SEEDS.set_index(["weighting", "type", "run"]).sort_index().to_string(float_format=lambda x: f"{x:.1f}"))
rows = []
for run, F in list(RUNS.items()) + [("best of 16", BEST16)]:
    for wname in "ab":
        _, kw, C11, _ = fig11c(F, wname)
        rows.append(dict(run=run if isinstance(run, str) else f"seed {run}", weighting=wname,
                         table4_within=int(table4(F, wname).within.sum()),
                         cck_tau_r_udr_fac=F[("CCK+BC to PC", wname, "UDR dep+fac")]["tau_r"],
                         **{f"kw_p_{q}": v for q, v in kw.items()}, posthoc_holding=int(C11.holds.sum())))
print("\nTable 4 values within tolerance (of 13), comprehensive-model CCK+BC tau_r (s), fig11c Kruskal-Wallis p and post hoc comparisons that hold (of 6):")
print(pd.DataFrame(rows).to_string(float_format=lambda x: f"{x:.4g}"))


**Result: fig11-model-pv-udr partially reproduced.** For the three PV+ types the lowest BIC is the UDR
depressing model under both readings, and adding facilitation to it lowers the error by 0.1 to 7.9%. The
decrease of the summed squared error with UDR is smaller than reported: 22.9%, 13.1% and 24.7% for PV+BC
to PV+BC, PV+BC to PC and AAC to PC under reading a (17.0, 8.2 and 21.2% under b), against 35, 22 and 34%.
Under reading a all three lie within 10 to 50%.

**Result: fig11-model-cck-facilitation reproduced.** Under reading b the lowest BIC for CCK+BC to PC is the
TM depressing plus facilitating model (-1389 against -1387 for UDR with facilitation, -1260 for TM
depressing), and its error is below that of the TM depressing model (15.6 against 22.5). Under reading a
the UDR model with facilitation has a marginally lower BIC (3497 against 3500). The margin under reading b
is 1.8, small for a BIC difference.

**Result: table4-model-parameters partially reproduced.** Under reading a 8 of the 13 values are within
tolerance (7 under reading b): Imax for all four types, tau_r for PV+BC to PV+BC (1.087 s) and CCK+BC to
PC (0.0655 s), U0 for PV+BC to PC and CCK+BC to PC. Outside: U0 of PV+BC to PV+BC (0.42) and AAC to PC
(0.30), tau_r of PV+BC to PC (1.22 s) and AAC to PC (1.93 s), and tau_f of CCK+BC to PC (40.2 against 10.9
ms). The comprehensive model's CCK+BC tau_r (variant b) is 0.415 s (reading b) or diverges (2.95e+10 s,
reading a), not 12.9 s.

**Result: fig11c-parameter-differences not reproduced.** In the comprehensive model U0 (p 0.0002, 0.0072)
and tau_f (p 0.026, 0.0001) differ across types but Imax does not (p 0.077, 0.162). U0 is smaller for
CCK+BC to PC than for PV+BC to PV+BC and PV+BC to PC under both readings; tau_f is larger for CCK+BC to
PC only against PV+BC to PV+BC under reading b, and Imax of PV+BC to PC is not smaller than that of the
other types.

**Result: fig11-model-prediction reproduced.** Under reading b, with only Imax refitted on the RIP and EPI
amplitudes, the lowest test error is the UDR depressing model for PV+BC to PC (324.0 pA²) and AAC to PC
(953.1 pA²), and the TM depressing plus facilitating model for CCK+BC to PC (3658.2 pA²). PV+BC to PV+BC
pairs have no RIP or EPI recordings, so that type is not tested. Under reading a the lowest test error is
the UDR model with facilitation for PV+BC to PC (317.9 pA²), which the criterion also counts, but the TM
depressing model for AAC to PC (994.4 against 1003.0 pA²). The TM depressing fit for CCK+BC to PC under
reading a gives an infinite test error in this run; it is 3765.7 pA² with the other seeds and with the
best of 16 starting points below.

**Run-to-run variation.** Four starting points do not always reach the same fit: with the generator
seeded 1, 2 or 3 the fits differ in places from the run above (seed 11). In seed 2 the UDR depressing fit
for PV+BC to PV+BC under reading a stops at a worse optimum (SSE decrease 4.5%, lowest BIC UDR with
facilitation), and in seed 3 the TM depressing plus facilitating fit for CCK+BC to PC under reading b does
(BIC -1341, so UDR with facilitation has the lowest BIC); in seeds 1 and 2 the CCK+BC result is as above.
Keeping for each model the best of all 16 starting points gives the same conclusions as seed 11: lowest
BIC UDR depressing for the three PV+ types (decreases 22.9, 13.1, 24.7% under a), TM depressing plus
facilitating for CCK+BC to PC under b (-1389.1 against -1387.3), 8 of 13 Table 4 values within tolerance
under a (7 under b), the Kruskal-Wallis part of fig11c failing under both readings (under a Imax gives p
0.038 but tau_f p 0.226; under b Imax gives p 0.495) with 2 of 6 post hoc differences holding, and the
same lowest test errors under b. The comprehensive model's CCK+BC tau_r under reading b is 2.3 s in seeds
1 to 3 and in the best of 16 (0.415 s above), still far from 12.9 s. The p values of fig11c move between
runs (tau_f 0.026 to 0.495 under a), and in no run do all three Kruskal-Wallis tests give p < 0.05. No
outcome changes, but fig11-model-cck-facilitation rests on a small BIC margin that a poorer optimum
reverses.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig4b-in-pc-kruskal-wallis | reproduced (assumed) |
| fig4b-in-pc-posthoc | not reproduced (assumed) |
| fig4c-cckdti-pc-silent-single-ap | not reproduced |
| fig5c-pv-rip-depression | reproduced (assumed) |
| fig5c-cck-rip-facilitation | reproduced (assumed) |
| fig5db-recovery | reproduced (assumed) |
| fig6-pv-trains-depression | not reproduced (assumed) |
| fig6-cckbc-pc-trains | not reproduced (assumed) |
| fig6-cckbc-async-onset | not reproduced (assumed) |
| cckdti-pc-transmission | not reproduced |
| cckdti-async-onset | not reproduced |
| async-charge-epi-vs-160 | not reproduced (assumed) |
| decay-epi-vs-160 | partially reproduced (assumed) |
| fig8-pvbc-pvbc-uipsc | partially reproduced |
| fig9-pvbc-pvbc-depression | reproduced |
| fig9-pvbc-pvbc-transmission-probability | not reproduced |
| fig9-pvbc-pvbc-recovery-11th | reproduced |
| fig10-cck-cck-heterogeneity | not reproduced |
| fig10-cck-cck-frequency | reproduced |
| fig10-cck-cck-async | not reproduced |
| fig11-model-pv-udr | partially reproduced (assumed) |
| fig11-model-cck-facilitation | reproduced (assumed) |
| table4-model-parameters | partially reproduced (assumed) |
| fig11c-parameter-differences | not reproduced (assumed) |
| fig11-model-prediction | reproduced (assumed) |
| fig13-pvbc-pc-recovery-halftime | reproduced (assumed) |
| fig8-cck-pairs-uipsc, fig8-cckbc-vs-cckdti-pairs, decay-pc-vs-interneuron | blocked: data missing (no single-AP recordings of the CCK+ pairs) |
| fig10bd-be-async-charge | blocked: data missing (no 320 Hz trains of the CCK+ pairs) |
| fig7a-cscl-kinetics, fig7b-cscl-stp | blocked: metadata insufficient |
| fig2b (3 claims), table1, table2, fig3 (3 claims), fig12, sipsc-continuum, fig13a, fig13c | blocked: data missing |

"Assumed" marks the outcomes on IN to PC pairs, which rest on the assumption that every such pair in the
data set was recorded with high KCl in both cells.

How the analysis differs from the paper's:

- The data set holds fewer pairs than the paper reports for most types (above), and no current-step,
  connectivity, spontaneous IPSC or field potential data.
- All IN to PC pairs are treated as high KCl pairs; the files do not say, and the per-pair check of
  holding current, rise time and latency shows no split but cannot rule out a mixture. In CCK+BC to PC
  pairs latency rises with the holding current, and with only the half of these pairs with the more
  negative holding current the fig4b differences in rise and latency are not significant.
- The model fits stop at different optima from different starting points. Keeping the best of 16
  starting points gives the conclusions of the run shown; the CCK+BC preference for the depressing and
  facilitating model rests on a BIC margin of 1.8.
- The interneuron pairs have no RIP recordings and the CCK+ pairs no 1 Hz train either; the first AP of
  the trains (reading d) is shown for them and decides nothing.
- Rise, latency, decay and peak are measured on averaged sweeps with standard choices of windows (baseline
  4 to 0.8 ms before the AP peak; response 0.3 to 8 ms after; decay fits over 20 and 50 ms) that the paper
  does not give.
- Both onset rules of the claims file mark asynchronous release in the PV+ pairs the paper calls
  synchronous, and in the 160 Hz trains of nearly every pair; the onset claims are judged on them and
  fail. The authors' undocumented `AR` marks agree with the paper's onsets and are shown only.
- The total charge window "at least 500 ms after the stimulation" does not fit in the sweeps; the
  pre-stimulation baseline is used.

What was found about the data set: the protocol timings differ from the descriptor and the paper (RIP at
222 Hz, EPI at 147 and 303 Hz, single AP 1.3 s before the bursts); `Experiments_description.txt` mislabels
two recordings, names eight files that are not in the zip and numbers the pairs differently from the
folders; three recordings have no sweeps; the `_EPA` tables use an undocumented code `AR`, leave sweeps
out without a note, and two hold labels in place of mean values; the recovery intervals' order is not
documented; there are no animal, slice or pipette-solution labels.